In [3]:
# ============================================================
# CUHK-X SMALL MODEL TRACK
# FINAL MULTIMODAL FUSION
#
# Modalities:
#   Skeleton + IR + Depth_Color + Radar
#
# Strategy:
#   Exact pretrained encoders
#   → cached embeddings
#   → learned feature-level fusion
#   → ONE 40-class prediction
# ============================================================

from pathlib import Path
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())

    for i in range(torch.cuda.device_count()):
        print(
            f"GPU {i}:",
            torch.cuda.get_device_name(i)
        )

# ------------------------------------------------------------
# Dataset paths
# ------------------------------------------------------------

DATA_ROOT = Path(
    "/kaggle/input/datasets/samasiayushman/"
    "small-model-track/Training/Training/data"
)

TEST_ROOT = Path(
    "/kaggle/input/datasets/samasiayushman/"
    "small-model-track/Testing/Testing/"
    "small_model_track_test-007/small_model_track_test"
)

# ------------------------------------------------------------
# Modality roots
# ------------------------------------------------------------

SKELETON_ROOT = DATA_ROOT / "Skeleton"
IR_ROOT       = DATA_ROOT / "IR"
DEPTH_ROOT    = DATA_ROOT / "Depth_Color"
RADAR_ROOT   = DATA_ROOT / "Radar"

# ------------------------------------------------------------
# Checkpoint directory
# ------------------------------------------------------------

RESULTS_ROOT = Path(
    "/kaggle/input/datasets/samasiayushman/"
    "ir-skeleton-model/results (1)"
)

SKEL_CKPT = Path(
    "/kaggle/input/datasets/samasiayushman/"
    "ir-skeleton-model/results(1)/"
    "BI_LSTM_BEST/bilstm_best_weights.pth"
)

IR_CKPT = Path(
    "/kaggle/input/datasets/samasiayushman/"
    "ir-skeleton-model/results/IR_V2_BEST/"
    "ir_v2_fusion_encoder.pth"
)

DEPTH_CKPT = (
    RESULTS_ROOT /
    "depth_color_v1_best.pt"
)

RADAR_CKPT = (
    RESULTS_ROOT /
    "radar_pointnet_tcn_best.pt"
)

# ------------------------------------------------------------
# General configuration
# ------------------------------------------------------------

NUM_CLASSES = 40

# Skeleton
SKELETON_SEQ_LEN = 64

# IR
IR_SEQ_LEN = 24
IR_W = 160
IR_H = 120

# Depth
DEPTH_MAX_FRAMES = 32
DEPTH_IMG_SIZE = 160

# Radar
RADAR_MAX_FRAMES = 128
RADAR_MAX_DETECTIONS = 32

# ------------------------------------------------------------
# Fusion
# ------------------------------------------------------------

FUSION_DIM = 512
FUSION_HEADS = 8
FUSION_LAYERS = 3

# ------------------------------------------------------------
# Cache/output directories
# ------------------------------------------------------------

WORK_ROOT = Path("/kaggle/working")

CACHE_ROOT = WORK_ROOT / "cuhk_x_fusion_cache"
CACHE_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_ROOT = WORK_ROOT / "cuhk_x_final"
OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print()
print("=" * 70)
print("CONFIGURATION")
print("=" * 70)

print("DATA_ROOT :", DATA_ROOT)
print("TEST_ROOT :", TEST_ROOT)

print("\nCheckpoints:")

for name, path in [
    ("Skeleton", SKEL_CKPT),
    ("IR", IR_CKPT),
    ("Depth", DEPTH_CKPT),
    ("Radar", RADAR_CKPT),
]:
    print(
        f"{name:10s}:",
        path,
        "✓" if path.exists() else "✗ MISSING"
    )

print("\nCache:", CACHE_ROOT)
print("Output:", OUTPUT_ROOT)

Device: cuda
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4

CONFIGURATION
DATA_ROOT : /kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data
TEST_ROOT : /kaggle/input/datasets/samasiayushman/small-model-track/Testing/Testing/small_model_track_test-007/small_model_track_test

Checkpoints:
Skeleton  : /kaggle/input/datasets/samasiayushman/ir-skeleton-model/results(1)/BI_LSTM_BEST/bilstm_best_weights.pth ✓
IR        : /kaggle/input/datasets/samasiayushman/ir-skeleton-model/results/IR_V2_BEST/ir_v2_fusion_encoder.pth ✓
Depth     : /kaggle/input/datasets/samasiayushman/ir-skeleton-model/results (1)/depth_color_v1_best.pt ✓
Radar     : /kaggle/input/datasets/samasiayushman/ir-skeleton-model/results (1)/radar_pointnet_tcn_best.pt ✓

Cache: /kaggle/working/cuhk_x_fusion_cache
Output: /kaggle/working/cuhk_x_final


In [2]:
from pathlib import Path
import torch
import numpy as np

DEPTH_CKPT = Path(
    "/kaggle/input/datasets/samasiayushman/"
    "ir-skeleton-model/results (1)/depth_color_v1_best.pt"
)

RADAR_CKPT = Path(
    "/kaggle/input/datasets/samasiayushman/"
    "ir-skeleton-model/results (1)/radar_pointnet_tcn_best.pt"
)


def inspect_checkpoint(path, name):
    print("\n" + "="*80)
    print(name)
    print("="*80)
    print("Path:", path)
    print("Size:", path.stat().st_size / 1024**2, "MB")

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    print("\nCheckpoint type:")
    print(type(ckpt))

    if isinstance(ckpt, dict):
        print("\nTop-level keys:")
        for k, v in ckpt.items():
            if isinstance(v, dict):
                print(f"  {k}: dict ({len(v)} keys)")
            elif torch.is_tensor(v):
                print(f"  {k}: tensor {tuple(v.shape)}")
            elif isinstance(v, (list, tuple)):
                print(f"  {k}: {type(v).__name__} len={len(v)}")
            else:
                print(f"  {k}: {type(v).__name__} = {str(v)[:200]}")

        if "model_state_dict" in ckpt:
            sd = ckpt["model_state_dict"]

            print("\nMODEL STATE DICT")
            print("Number of keys:", len(sd))

            for k, v in sd.items():
                if torch.is_tensor(v):
                    print(f"  {k:60s} {tuple(v.shape)}")
                else:
                    print(f"  {k:60s} {type(v)}")

        elif all(torch.is_tensor(v) for v in ckpt.values()):
            print("\nLooks like raw state_dict")

    return ckpt


depth_ckpt = inspect_checkpoint(
    DEPTH_CKPT,
    "DEPTH COLOR V1"
)

radar_ckpt = inspect_checkpoint(
    RADAR_CKPT,
    "RADAR POINTNET + TCN"
)


DEPTH COLOR V1
Path: /kaggle/input/datasets/samasiayushman/ir-skeleton-model/results (1)/depth_color_v1_best.pt
Size: 3.5794858932495117 MB

Checkpoint type:
<class 'dict'>

Top-level keys:
  model_state: dict (142 keys)
  epoch: int = 6
  val_acc: float = 0.18759018759018758
  val_balanced_acc: float64 = 0.14694974359967966
  val_macro_f1: float = 0.08425613294098123
  num_classes: int = 40
  img_size: int = 160
  max_frames: int = 32

RADAR POINTNET + TCN
Path: /kaggle/input/datasets/samasiayushman/ir-skeleton-model/results (1)/radar_pointnet_tcn_best.pt
Size: 6.645051002502441 MB

Checkpoint type:
<class 'dict'>

Top-level keys:
  model_state_dict: dict (86 keys)
  feature_mean: ndarray = [-1.73170313e-01  1.08870163e+00 -1.68657074e-01 -7.11193094e-04
  1.73842025e+02  5.26117348e+02]
  feature_std: ndarray = [ 0.88422244  1.18826219  1.24050746  0.15830402 45.52745306 65.53307942]
  max_frames: int = 128
  max_detections: int = 32
  val_acc: float = 0.018808777429467086
  val_f1:

In [4]:
# ============================================================
# CELL 1 — CHECKPOINT LOADING HELPERS
# ============================================================

def load_raw_state_dict(path):
    """
    Load a trusted raw state_dict checkpoint.
    """

    obj = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    if not isinstance(obj, dict):
        raise TypeError(
            f"Expected dict/state_dict, got {type(obj)}"
        )

    return obj


def load_full_checkpoint(path):
    """
    Load trusted full training checkpoints.

    These checkpoints contain numpy metadata,
    therefore weights_only=False is required.
    """

    obj = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    if not isinstance(obj, dict):
        raise TypeError(
            f"Expected checkpoint dict, got {type(obj)}"
        )

    return obj


print("Checkpoint helper functions ready.")

Checkpoint helper functions ready.


In [5]:
# ============================================================
# CELL 2 — CANONICAL MULTIMODAL TRIAL INDEX
# ============================================================

records = []

for action_dir in sorted(SKELETON_ROOT.iterdir()):

    if not action_dir.is_dir():
        continue

    action_name = action_dir.name

    try:
        class_id = int(
            action_name.split("_", 1)[0]
        )
    except Exception:
        continue

    for user_dir in sorted(action_dir.iterdir()):

        if not user_dir.is_dir():
            continue

        user = user_dir.name

        for trial_dir in sorted(user_dir.iterdir()):

            if not trial_dir.is_dir():
                continue

            records.append({
                "action": action_name,
                "class_id": class_id,
                "user": user,
                "trial": trial_dir.name,
                "skeleton_path": str(trial_dir),
            })


common_df = pd.DataFrame(records)

print("Skeleton trials:", len(common_df))
print("Classes:", common_df["class_id"].nunique())
print("Users:", common_df["user"].nunique())

common_df.head()

Skeleton trials: 2931
Classes: 40
Users: 18


,action,class_id,user,trial,skeleton_path
0,0_Wash_face,0,user16,1-1-1,/kaggle/input/datasets/samasiayushman/small-mo...
1,0_Wash_face,0,user16,1-1-2,/kaggle/input/datasets/samasiayushman/small-mo...
2,0_Wash_face,0,user16,1-1-3,/kaggle/input/datasets/samasiayushman/small-mo...
3,0_Wash_face,0,user18,7-1-1,/kaggle/input/datasets/samasiayushman/small-mo...
4,0_Wash_face,0,user18,7-1-2,/kaggle/input/datasets/samasiayushman/small-mo...


In [6]:
# ============================================================
# CELL 3 — CANONICAL CROSS-SUBJECT SPLIT
# ============================================================

from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        common_df,
        groups=common_df["user"]
    )
)

train_df = (
    common_df.iloc[train_idx]
    .reset_index(drop=True)
)

val_df = (
    common_df.iloc[val_idx]
    .reset_index(drop=True)
)

print("=" * 70)
print("CANONICAL FUSION SPLIT")
print("=" * 70)

print("Total :", len(common_df))
print("Train :", len(train_df))
print("Val   :", len(val_df))

print("\nTrain users:")
print(
    sorted(
        train_df["user"].unique(),
        key=lambda x: int(x.replace("user", ""))
    )
)

print("\nVal users:")
print(
    sorted(
        val_df["user"].unique(),
        key=lambda x: int(x.replace("user", ""))
    )
)

print("\nClass counts:")
print(
    "Train classes:",
    train_df["class_id"].nunique()
)

print(
    "Val classes:",
    val_df["class_id"].nunique()
)

CANONICAL FUSION SPLIT
Total : 2931
Train : 2238
Val   : 693

Train users:
['user3', 'user4', 'user5', 'user6', 'user7', 'user8', 'user9', 'user17', 'user18', 'user19', 'user20', 'user21', 'user23', 'user24']

Val users:
['user1', 'user2', 'user16', 'user22']

Class counts:
Train classes: 40
Val classes: 40


In [7]:
# ============================================================
# CELL 4 — SKELETON CHECKPOINT INSPECTION
# ============================================================

skel_state = load_raw_state_dict(SKEL_CKPT)

print("=" * 70)
print("SKELETON CHECKPOINT")
print("=" * 70)

print("Keys:", len(skel_state))

for k in list(skel_state.keys())[:10]:
    print(
        f"{k:50s}",
        tuple(skel_state[k].shape)
    )

print("\nLast keys:")

for k in list(skel_state.keys())[-10:]:
    print(
        f"{k:50s}",
        tuple(skel_state[k].shape)
    )

print("\nClassifier present:",
      any(k.startswith("classifier.") for k in skel_state))

print(
    "Expected encoder embedding: [B, 256]"
)

SKELETON CHECKPOINT
Keys: 34
input_projection.0.weight                          (128, 204)
input_projection.0.bias                            (128,)
input_projection.1.weight                          (128,)
input_projection.1.bias                            (128,)
lstm.weight_ih_l0                                  (512, 128)
lstm.weight_hh_l0                                  (512, 128)
lstm.bias_ih_l0                                    (512,)
lstm.bias_hh_l0                                    (512,)
lstm.weight_ih_l0_reverse                          (512, 128)
lstm.weight_hh_l0_reverse                          (512, 128)

Last keys:
norm.weight                                        (256,)
norm.bias                                          (256,)
frame_attention.0.weight                           (128, 256)
frame_attention.0.bias                             (128,)
frame_attention.2.weight                           (1, 128)
frame_attention.2.bias                             (1,)
classif

In [10]:
# ============================================================
# CELL 5 — IR CHECKPOINT INSPECTION
# ============================================================

ir_state = load_raw_state_dict(IR_CKPT)

print("=" * 70)
print("IR FUSION ENCODER")
print("=" * 70)

print("Keys:", len(ir_state))

for k in list(ir_state.keys())[:15]:
    print(
        f"{k:55s}",
        tuple(ir_state[k].shape)
    )

print("\n")

print(
    "Classifier present:",
    any(
        k.startswith("classifier.")
        for k in ir_state.keys()
    )
)

print(
    "Expected IR embedding: [B, 512]"
)

IR FUSION ENCODER
Keys: 174
encoder.stem.0.weight                                   (32, 3, 5, 5)
encoder.stem.1.weight                                   (32,)
encoder.stem.1.bias                                     (32,)
encoder.stem.1.running_mean                             (32,)
encoder.stem.1.running_var                              (32,)
encoder.stem.1.num_batches_tracked                      ()
encoder.layer1.0.conv1.weight                           (32, 32, 3, 3)
encoder.layer1.0.bn1.weight                             (32,)
encoder.layer1.0.bn1.bias                               (32,)
encoder.layer1.0.bn1.running_mean                       (32,)
encoder.layer1.0.bn1.running_var                        (32,)
encoder.layer1.0.bn1.num_batches_tracked                ()
encoder.layer1.0.conv2.weight                           (32, 32, 3, 3)
encoder.layer1.0.bn2.weight                             (32,)
encoder.layer1.0.bn2.bias                               (32,)


Classifier present: 

In [8]:
# ============================================================
# CELL 6 — DEPTH CHECKPOINT INSPECTION
# ============================================================

depth_ckpt = load_full_checkpoint(DEPTH_CKPT)

depth_state = depth_ckpt["model_state"]

print("=" * 70)
print("DEPTH + COLOR CHECKPOINT")
print("=" * 70)

print("Epoch:", depth_ckpt["epoch"])
print("Val accuracy:", depth_ckpt["val_acc"])
print("Balanced accuracy:", depth_ckpt["val_balanced_acc"])
print("Macro F1:", depth_ckpt["val_macro_f1"])

print("\nConfig:")
print("Classes:", depth_ckpt["num_classes"])
print("Image size:", depth_ckpt["img_size"])
print("Max frames:", depth_ckpt["max_frames"])

print("\nState dict keys:", len(depth_state))

print("\nFirst keys:")

for k in list(depth_state.keys())[:12]:
    print(
        f"{k:50s}",
        tuple(depth_state[k].shape)
    )

print("\nExpected encoder embedding: [B, 128]")

DEPTH + COLOR CHECKPOINT
Epoch: 6
Val accuracy: 0.18759018759018758
Balanced accuracy: 0.14694974359967966
Macro F1: 0.08425613294098123

Config:
Classes: 40
Image size: 160
Max frames: 32

State dict keys: 142

First keys:
cnn.net.0.conv1.weight                             (32, 3, 3, 3)
cnn.net.0.bn1.weight                               (32,)
cnn.net.0.bn1.bias                                 (32,)
cnn.net.0.bn1.running_mean                         (32,)
cnn.net.0.bn1.running_var                          (32,)
cnn.net.0.bn1.num_batches_tracked                  ()
cnn.net.0.conv2.weight                             (32, 32, 3, 3)
cnn.net.0.bn2.weight                               (32,)
cnn.net.0.bn2.bias                                 (32,)
cnn.net.0.bn2.running_mean                         (32,)
cnn.net.0.bn2.running_var                          (32,)
cnn.net.0.bn2.num_batches_tracked                  ()

Expected encoder embedding: [B, 128]


In [9]:
# ============================================================
# CELL 7 — RADAR CHECKPOINT INSPECTION
# ============================================================

radar_ckpt = load_full_checkpoint(RADAR_CKPT)

radar_state = radar_ckpt["model_state_dict"]

print("=" * 70)
print("RADAR CHECKPOINT")
print("=" * 70)

print("Epoch:", radar_ckpt["epoch"])
print("Val accuracy:", radar_ckpt["val_acc"])
print("Val F1:", radar_ckpt["val_f1"])

print("\nRadar normalization:")

print("mean =", radar_ckpt["feature_mean"])
print("std  =", radar_ckpt["feature_std"])

print("\nSequence configuration:")
print("Max frames:", radar_ckpt["max_frames"])
print("Max detections:", radar_ckpt["max_detections"])

print("\nState dict keys:", len(radar_state))

print("\nFirst keys:")

for k in list(radar_state.keys())[:15]:
    print(
        f"{k:55s}",
        tuple(radar_state[k].shape)
    )

print("\nExpected encoder embedding: [B, 256]")

RADAR CHECKPOINT
Epoch: 1
Val accuracy: 0.018808777429467086
Val F1: 0.000997920997920998

Radar normalization:
mean = [-1.73170313e-01  1.08870163e+00 -1.68657074e-01 -7.11193094e-04
  1.73842025e+02  5.26117348e+02]
std  = [ 0.88422244  1.18826219  1.24050746  0.15830402 45.52745306 65.53307942]

Sequence configuration:
Max frames: 128
Max detections: 32

State dict keys: 86

First keys:
point_encoder.mlp.0.weight                              (64, 6)
point_encoder.mlp.0.bias                                (64,)
point_encoder.mlp.1.weight                              (64,)
point_encoder.mlp.1.bias                                (64,)
point_encoder.mlp.1.running_mean                        (64,)
point_encoder.mlp.1.running_var                         (64,)
point_encoder.mlp.1.num_batches_tracked                 ()
point_encoder.mlp.3.weight                              (64, 64)
point_encoder.mlp.3.bias                                (64,)
point_encoder.mlp.4.weight                     

In [11]:
# ============================================================
# CELL 8 — INSPECT RADAR DIRECTORY STRUCTURE
# ============================================================

print("=" * 70)
print("RADAR DIRECTORY STRUCTURE")
print("=" * 70)

print("RADAR_ROOT:")
print(RADAR_ROOT)

print("\nExists:", RADAR_ROOT.exists())

# Show first few levels
if RADAR_ROOT.exists():

    for p in list(RADAR_ROOT.rglob("*"))[:80]:
        print(p)

else:
    print("Radar root does not exist at this path.")

RADAR DIRECTORY STRUCTURE
RADAR_ROOT:
/kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Radar

Exists: True
/kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Radar/29_Do_squats
/kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Radar/32_Stand_up
/kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Radar/6_Drink_water
/kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Radar/14_Wipe_bowls
/kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Radar/4_Wipe_hands
/kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Radar/23_Listen_to_music_with_headphones
/kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Radar/5_Put_on_clothes
/kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Radar/22_Turn_pages
/kaggle/input/datasets/samasiayushman/small-model-t

In [12]:
# ============================================================
# CELL 9 — RADAR CSV PATH / FILENAME INSPECTION
# ============================================================

radar_csvs = sorted(RADAR_ROOT.rglob("*.csv"))

print("=" * 70)
print("RADAR CSV INSPECTION")
print("=" * 70)

print("Total CSV files:", len(radar_csvs))

for p in radar_csvs[:30]:
    print(p)

RADAR CSV INSPECTION
Total CSV files: 2914
/kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Radar/0_Wash_face/user16/1-1-1/radar_output_T2025-06-10_10-43-36.157.csv
/kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Radar/0_Wash_face/user16/1-1-2/radar_output_T2025-06-10_10-45-13.133.csv
/kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Radar/0_Wash_face/user16/1-1-3/radar_output_T2025-06-10_10-46-32.242.csv
/kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Radar/0_Wash_face/user18/7-1-1/radar_output_T2025-06-10_11-29-05.077.csv
/kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Radar/0_Wash_face/user18/7-1-2/radar_output_T2025-06-10_11-31-09.379.csv
/kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Radar/0_Wash_face/user18/7-1-3/radar_output_T2025-06-10_11-33-10.134.csv
/kaggle/input/datasets/samasiayushman/small-m

In [13]:
# ============================================================
# CELL 10 — RADAR PATH COMPONENT ANALYSIS
# ============================================================

print("=" * 70)
print("RADAR PATH COMPONENTS")
print("=" * 70)

for p in radar_csvs[:20]:

    relative = p.relative_to(RADAR_ROOT)

    print("\nFULL :", p)
    print("REL  :", relative)
    print("PARTS:", relative.parts)
    print("NAME :", p.name)

RADAR PATH COMPONENTS

FULL : /kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Radar/0_Wash_face/user16/1-1-1/radar_output_T2025-06-10_10-43-36.157.csv
REL  : 0_Wash_face/user16/1-1-1/radar_output_T2025-06-10_10-43-36.157.csv
PARTS: ('0_Wash_face', 'user16', '1-1-1', 'radar_output_T2025-06-10_10-43-36.157.csv')
NAME : radar_output_T2025-06-10_10-43-36.157.csv

FULL : /kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Radar/0_Wash_face/user16/1-1-2/radar_output_T2025-06-10_10-45-13.133.csv
REL  : 0_Wash_face/user16/1-1-2/radar_output_T2025-06-10_10-45-13.133.csv
PARTS: ('0_Wash_face', 'user16', '1-1-2', 'radar_output_T2025-06-10_10-45-13.133.csv')
NAME : radar_output_T2025-06-10_10-45-13.133.csv

FULL : /kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Radar/0_Wash_face/user16/1-1-3/radar_output_T2025-06-10_10-46-32.242.csv
REL  : 0_Wash_face/user16/1-1-3/radar_output_T2025-06-10_10-46-32.242.csv
PART

In [14]:
# ============================================================
# CELL 11 — RADAR TRIAL INDEX
# ============================================================

radar_records = []

for csv_path in sorted(RADAR_ROOT.rglob("*.csv")):

    parts = csv_path.relative_to(RADAR_ROOT).parts

    # Expected:
    # action / user / trial / filename.csv

    if len(parts) < 4:
        continue

    action_name = parts[0]
    user = parts[1]
    trial = parts[2]

    try:
        class_id = int(
            action_name.split("_", 1)[0]
        )
    except Exception:
        continue

    radar_records.append({
        "action": action_name,
        "class_id": class_id,
        "user": user,
        "trial": trial,
        "radar_path": str(csv_path),
    })

radar_df = pd.DataFrame(radar_records)

print("=" * 70)
print("RADAR INDEX")
print("=" * 70)

print("Radar records:", len(radar_df))
print("Classes:", radar_df["class_id"].nunique())
print("Users:", radar_df["user"].nunique())

print("\nUsers:")
print(
    sorted(
        radar_df["user"].unique(),
        key=lambda x: int(x.replace("user", ""))
    )
)

print("\nDuplicate trial keys:")

duplicates = (
    radar_df
    .groupby(["user", "trial", "class_id"])
    .size()
)

print(
    duplicates[duplicates > 1].head(20)
)

radar_df.head()

RADAR INDEX
Radar records: 2914
Classes: 40
Users: 18

Users:
['user1', 'user2', 'user3', 'user4', 'user5', 'user6', 'user7', 'user8', 'user9', 'user16', 'user17', 'user18', 'user19', 'user20', 'user21', 'user22', 'user23', 'user24']

Duplicate trial keys:
Series([], dtype: int64)


,action,class_id,user,trial,radar_path
0,0_Wash_face,0,user16,1-1-1,/kaggle/input/datasets/samasiayushman/small-mo...
1,0_Wash_face,0,user16,1-1-2,/kaggle/input/datasets/samasiayushman/small-mo...
2,0_Wash_face,0,user16,1-1-3,/kaggle/input/datasets/samasiayushman/small-mo...
3,0_Wash_face,0,user18,7-1-1,/kaggle/input/datasets/samasiayushman/small-mo...
4,0_Wash_face,0,user18,7-1-2,/kaggle/input/datasets/samasiayushman/small-mo...


In [15]:
# ============================================================
# CELL 12 — SKELETON + RADAR ALIGNMENT
# ============================================================

alignment_keys = [
    "user",
    "trial",
    "class_id"
]

aligned_df = common_df.merge(
    radar_df[
        alignment_keys + ["radar_path"]
    ],
    on=alignment_keys,
    how="left",
    validate="one_to_one"
)

print("=" * 70)
print("SKELETON + RADAR ALIGNMENT")
print("=" * 70)

print("Total trials:", len(aligned_df))

print(
    "Radar present:",
    aligned_df["radar_path"].notna().sum()
)

print(
    "Radar missing:",
    aligned_df["radar_path"].isna().sum()
)

print(
    "Radar coverage:",
    f"{aligned_df['radar_path'].notna().mean()*100:.3f}%"
)

print("\nMissing Radar examples:")

print(
    aligned_df[
        aligned_df["radar_path"].isna()
    ][
        ["action", "class_id", "user", "trial"]
    ].head(30)
)

SKELETON + RADAR ALIGNMENT
Total trials: 2931
Radar present: 2914
Radar missing: 17
Radar coverage: 99.420%

Missing Radar examples:
                          action  class_id    user  trial
44                10_Stir_drinks        10   user1  2-1-1
469   15_Wipe_windows_and_tables        15   user8  3-3-3
479              16_Fold_clothes        16   user2  1-1-1
480              16_Fold_clothes        16   user2  1-1-2
481              16_Fold_clothes        16   user2  1-1-3
1743                     36_Walk        36   user1  2-1-1
2043                     36_Walk        36   user8  3-3-3
2210    39_Take_body_temperature        39  user17  4-1-2
2211    39_Take_body_temperature        39  user17  4-1-3
2212    39_Take_body_temperature        39  user17  7-1-1
2213    39_Take_body_temperature        39  user17  7-1-2
2214    39_Take_body_temperature        39  user17  7-1-3
2408            5_Put_on_clothes         5   user3  1-1-1
2409            5_Put_on_clothes         5   user3  1-1

In [20]:
# ============================================================
# CELL 13 — CORRECT RADAR LABEL CONSISTENCY CHECK
# ============================================================

radar_label_check = radar_df.merge(
    common_df[
        ["user", "trial", "class_id"]
    ],
    on=[
        "user",
        "trial",
        "class_id"
    ],
    how="inner",
    validate="one_to_one"
)

print("=" * 70)
print("RADAR LABEL CONSISTENCY")
print("=" * 70)

print("Radar records:", len(radar_df))
print("Matched canonical records:", len(radar_label_check))

print(
    "Unmatched Radar records:",
    len(radar_df) - len(radar_label_check)
)

if len(radar_label_check) == len(radar_df):
    print("\n✓ ALL RADAR RECORDS MATCH CANONICAL TRIALS")
else:
    print("\n⚠ Some Radar records do not match.")
    
    unmatched = radar_df.merge(
        common_df[
            ["user", "trial", "class_id"]
        ],
        on=[
            "user",
            "trial",
            "class_id"
        ],
        how="left",
        indicator=True
    )

    display(
        unmatched[
            unmatched["_merge"] == "left_only"
        ].head(30)
    )

RADAR LABEL CONSISTENCY
Radar records: 2914
Matched canonical records: 2914
Unmatched Radar records: 0

✓ ALL RADAR RECORDS MATCH CANONICAL TRIALS


In [17]:
# ============================================================
# CELL 14 — DEPTH/COLOR TRIAL INDEX
# ============================================================

depth_records = []

if DEPTH_ROOT.exists():

    for action_dir in sorted(DEPTH_ROOT.iterdir()):

        if not action_dir.is_dir():
            continue

        action_name = action_dir.name

        try:
            class_id = int(
                action_name.split("_", 1)[0]
            )
        except Exception:
            continue

        for user_dir in sorted(action_dir.iterdir()):

            if not user_dir.is_dir():
                continue

            user = user_dir.name

            for trial_dir in sorted(user_dir.iterdir()):

                if not trial_dir.is_dir():
                    continue

                trial = trial_dir.name

                # Collect image files.
                files = sorted(
                    list(trial_dir.glob("*.png"))
                    + list(trial_dir.glob("*.jpg"))
                    + list(trial_dir.glob("*.jpeg"))
                )

                depth_records.append({
                    "action": action_name,
                    "class_id": class_id,
                    "user": user,
                    "trial": trial,
                    "depth_path": str(trial_dir),
                    "files": [str(x) for x in files],
                    "n_files": len(files),
                })

else:
    print("WARNING: DEPTH_ROOT does not exist:", DEPTH_ROOT)

depth_df = pd.DataFrame(depth_records)

print("=" * 70)
print("DEPTH/COLOR INDEX")
print("=" * 70)

print("Depth trials:", len(depth_df))
print("Classes:", depth_df["class_id"].nunique())
print("Users:", depth_df["user"].nunique())

print("\nFirst rows:")
display(depth_df.head())

DEPTH/COLOR INDEX
Depth trials: 2931
Classes: 40
Users: 18

First rows:


,action,class_id,user,trial,depth_path,files,n_files
0,0_Wash_face,0,user16,1-1-1,/kaggle/input/datasets/samasiayushman/small-mo...,[/kaggle/input/datasets/samasiayushman/small-m...,47
1,0_Wash_face,0,user16,1-1-2,/kaggle/input/datasets/samasiayushman/small-mo...,[/kaggle/input/datasets/samasiayushman/small-m...,55
2,0_Wash_face,0,user16,1-1-3,/kaggle/input/datasets/samasiayushman/small-mo...,[/kaggle/input/datasets/samasiayushman/small-m...,22
3,0_Wash_face,0,user18,7-1-1,/kaggle/input/datasets/samasiayushman/small-mo...,[/kaggle/input/datasets/samasiayushman/small-m...,20
4,0_Wash_face,0,user18,7-1-2,/kaggle/input/datasets/samasiayushman/small-mo...,[/kaggle/input/datasets/samasiayushman/small-m...,13


In [19]:
# ============================================================
# CELL 15 — COMPLETE FOUR-MODALITY ALIGNMENT
# ============================================================

fusion_df = common_df.merge(
    depth_df[
        [
            "user",
            "trial",
            "class_id",
            "depth_path",
            "files",
            "n_files"
        ]
    ],
    on=[
        "user",
        "trial",
        "class_id"
    ],
    how="left",
    validate="one_to_one"
)

fusion_df = fusion_df.merge(
    radar_df[
        [
            "user",
            "trial",
            "class_id",
            "radar_path"
        ]
    ],
    on=[
        "user",
        "trial",
        "class_id"
    ],
    how="left",
    validate="one_to_one"
)

print("=" * 70)
print("FOUR-MODALITY FUSION INDEX")
print("=" * 70)

print("Total:", len(fusion_df))

print(
    "Skeleton:",
    fusion_df["skeleton_path"].notna().sum()
)

print(
    "Depth:",
    fusion_df["depth_path"].notna().sum()
)

print(
    "Radar:",
    fusion_df["radar_path"].notna().sum()
)

print(
    "IR:  will verify separately"
)

FOUR-MODALITY FUSION INDEX
Total: 2931
Skeleton: 2931
Depth: 2931
Radar: 2914
IR:  will verify separately


In [21]:
# ============================================================
# CELL 16 — IR TRIAL INDEX
# ============================================================

ir_records = []

for action_dir in sorted(IR_ROOT.iterdir()):

    if not action_dir.is_dir():
        continue

    action_name = action_dir.name

    try:
        class_id = int(
            action_name.split("_", 1)[0]
        )
    except Exception:
        continue

    for user_dir in sorted(action_dir.iterdir()):

        if not user_dir.is_dir():
            continue

        user = user_dir.name

        for trial_dir in sorted(user_dir.iterdir()):

            if not trial_dir.is_dir():
                continue

            trial = trial_dir.name

            png_files = sorted(
                trial_dir.glob("*.png")
            )

            ir_records.append({
                "action": action_name,
                "class_id": class_id,
                "user": user,
                "trial": trial,
                "ir_path": str(trial_dir),
                "ir_n_files": len(png_files),
            })

ir_df = pd.DataFrame(ir_records)

print("=" * 70)
print("IR INDEX")
print("=" * 70)

print("IR trials:", len(ir_df))
print("Classes:", ir_df["class_id"].nunique())
print("Users:", ir_df["user"].nunique())

print("\nEmpty IR trials:")
print(
    (ir_df["ir_n_files"] == 0).sum()
)

display(ir_df.head())

IR INDEX
IR trials: 2933
Classes: 40
Users: 18

Empty IR trials:
0


,action,class_id,user,trial,ir_path,ir_n_files
0,0_Wash_face,0,user16,1-1-1,/kaggle/input/datasets/samasiayushman/small-mo...,47
1,0_Wash_face,0,user16,1-1-2,/kaggle/input/datasets/samasiayushman/small-mo...,55
2,0_Wash_face,0,user16,1-1-3,/kaggle/input/datasets/samasiayushman/small-mo...,22
3,0_Wash_face,0,user18,7-1-1,/kaggle/input/datasets/samasiayushman/small-mo...,20
4,0_Wash_face,0,user18,7-1-2,/kaggle/input/datasets/samasiayushman/small-mo...,13


---
---

In [24]:
# ============================================================
# CELL 18 — FINAL MASTER DATASET
# ============================================================

fusion_df = fusion_df.copy()

# ------------------------------------------------------------
# Modality availability
# ------------------------------------------------------------

fusion_df["has_skeleton"] = (
    fusion_df["skeleton_path"].notna()
)

fusion_df["has_ir"] = (
    fusion_df["ir_path"].notna()
)

fusion_df["has_depth"] = (
    fusion_df["depth_path"].notna()
)

fusion_df["has_radar"] = (
    fusion_df["radar_path"].notna()
)

# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

assert fusion_df["has_skeleton"].all()
assert fusion_df["has_ir"].all()
assert fusion_df["has_depth"].all()

assert fusion_df["class_id"].between(
    0, NUM_CLASSES - 1
).all()

# ------------------------------------------------------------
# Apply canonical split
# ------------------------------------------------------------

train_keys = set(
    zip(
        train_df["user"],
        train_df["trial"],
        train_df["class_id"]
    )
)

val_keys = set(
    zip(
        val_df["user"],
        val_df["trial"],
        val_df["class_id"]
    )
)

fusion_train_df = fusion_df[
    fusion_df.apply(
        lambda r: (
            r["user"],
            r["trial"],
            r["class_id"]
        ) in train_keys,
        axis=1
    )
].reset_index(drop=True)

fusion_val_df = fusion_df[
    fusion_df.apply(
        lambda r: (
            r["user"],
            r["trial"],
            r["class_id"]
        ) in val_keys,
        axis=1
    )
].reset_index(drop=True)

print("=" * 70)
print("FINAL FUSION DATASET")
print("=" * 70)

print("Total:", len(fusion_df))
print("Train:", len(fusion_train_df))
print("Val:", len(fusion_val_df))

print("\nTrain Radar:")
print(
    fusion_train_df["has_radar"].value_counts()
)

print("\nVal Radar:")
print(
    fusion_val_df["has_radar"].value_counts()
)

print("\nTrain users:")
print(
    sorted(
        fusion_train_df["user"].unique(),
        key=lambda x: int(x.replace("user", ""))
    )
)

print("\nVal users:")
print(
    sorted(
        fusion_val_df["user"].unique(),
        key=lambda x: int(x.replace("user", ""))
    )
)

FINAL FUSION DATASET
Total: 2931
Train: 2238
Val: 693

Train Radar:
has_radar
True     2228
False      10
Name: count, dtype: int64

Val Radar:
has_radar
True     686
False      7
Name: count, dtype: int64

Train users:
['user3', 'user4', 'user5', 'user6', 'user7', 'user8', 'user9', 'user17', 'user18', 'user19', 'user20', 'user21', 'user23', 'user24']

Val users:
['user1', 'user2', 'user16', 'user22']


In [25]:
# ============================================================
# CELL 19 — SAVE MASTER INDEX
# ============================================================

INDEX_PATH = CACHE_ROOT / "fusion_master_index.csv"

fusion_df.to_csv(
    INDEX_PATH,
    index=False
)

print("Saved:", INDEX_PATH)
print("Rows:", len(fusion_df))

Saved: /kaggle/working/cuhk_x_fusion_cache/fusion_master_index.csv
Rows: 2931


In [26]:
# ============================================================
# CELL 20 — MODEL IMPORTS
# ============================================================

import math
import copy
import json
import warnings

from tqdm.auto import tqdm

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

PyTorch: 2.10.0+cu128
Device: cuda


In [28]:
# ============================================================
# CELL 21 — SKELETON ENCODER
# ============================================================

class S6SkeletonEncoder(nn.Module):

    def __init__(
        self,
        input_size=204,
        projection_size=128,
        hidden_size=128,
        num_layers=2,
        num_heads=4,
        dropout=0.3,
    ):
        super().__init__()

        self.input_projection = nn.Sequential(
            nn.Linear(input_size, projection_size),
            nn.LayerNorm(projection_size),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.lstm = nn.LSTM(
            input_size=projection_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True,
        )

        feature_size = hidden_size * 2

        self.temporal_attention = nn.MultiheadAttention(
            embed_dim=feature_size,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.norm = nn.LayerNorm(feature_size)

        self.frame_attention = nn.Sequential(
            nn.Linear(feature_size, 128),
            nn.Tanh(),
            nn.Linear(128, 1),
        )

        self.embedding_size = feature_size

    def forward(self, x):

        B, T, J, F = x.shape

        x = x.reshape(
            B,
            T,
            J * F
        )

        x = self.input_projection(x)

        x, _ = self.lstm(x)

        attended, _ = self.temporal_attention(
            x, x, x
        )

        x = self.norm(
            x + attended
        )

        scores = self.frame_attention(x)

        weights = torch.softmax(
            scores,
            dim=1
        )

        x = torch.sum(
            x * weights,
            dim=1
        )

        return x

# ============================================================
# CELL 22 — LOAD SKELETON ENCODER
# ============================================================

skeleton_encoder = S6SkeletonEncoder().to(DEVICE)

skel_encoder_state = {
    k: v
    for k, v in skel_state.items()
    if not k.startswith("classifier.")
}

missing, unexpected = skeleton_encoder.load_state_dict(
    skel_encoder_state,
    strict=True
)

print("Skeleton encoder loaded ✓")
print("Missing:", missing)
print("Unexpected:", unexpected)
print("Embedding size:", skeleton_encoder.embedding_size)

skeleton_encoder.eval()

Skeleton encoder loaded ✓
Missing: []
Unexpected: []
Embedding size: 256


S6SkeletonEncoder(
  (input_projection): Sequential(
    (0): Linear(in_features=204, out_features=128, bias=True)
    (1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.3, inplace=False)
  )
  (lstm): LSTM(128, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (temporal_attention): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
  )
  (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (frame_attention): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): Tanh()
    (2): Linear(in_features=128, out_features=1, bias=True)
  )
)

In [29]:
# ============================================================
# CELL 23 — IR ENCODER COMPONENTS
# ============================================================

class ResidualBlock(nn.Module):

    def __init__(
        self,
        in_ch,
        out_ch,
        stride=1
    ):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_ch,
            out_ch,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(out_ch)

        self.conv2 = nn.Conv2d(
            out_ch,
            out_ch,
            kernel_size=3,
            padding=1,
            bias=False
        )

        self.bn2 = nn.BatchNorm2d(out_ch)

        if (
            stride != 1
            or in_ch != out_ch
        ):
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_ch,
                    out_ch,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),
                nn.BatchNorm2d(out_ch)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):

        residual = self.shortcut(x)

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.gelu(x)

        x = self.conv2(x)
        x = self.bn2(x)

        x = x + residual

        x = F.gelu(x)

        return x


class IRFrameEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(
                3, 32,
                kernel_size=5,
                stride=2,
                padding=2,
                bias=False
            ),
            nn.BatchNorm2d(32),
            nn.GELU()
        )

        self.layer1 = nn.Sequential(
            ResidualBlock(32, 32),
            ResidualBlock(32, 32)
        )

        self.layer2 = nn.Sequential(
            ResidualBlock(
                32, 64,
                stride=2
            ),
            ResidualBlock(64, 64)
        )

        self.layer3 = nn.Sequential(
            ResidualBlock(
                64, 128,
                stride=2
            ),
            ResidualBlock(128, 128)
        )

        self.layer4 = nn.Sequential(
            ResidualBlock(
                128, 192,
                stride=2
            ),
            ResidualBlock(192, 192)
        )

        self.layer5 = nn.Sequential(
            ResidualBlock(
                192, 256,
                stride=2
            ),
            ResidualBlock(256, 256)
        )

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.projection = nn.Sequential(
            nn.Linear(256, 384),
            nn.LayerNorm(384),
            nn.GELU(),
            nn.Dropout(0.15)
        )

    def forward(self, x):

        x = self.stem(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)

        x = self.pool(x)

        x = x.flatten(1)

        x = self.projection(x)

        return x


class TemporalAttention(nn.Module):

    def __init__(self, dim):

        super().__init__()

        self.score = nn.Sequential(
            nn.Linear(
                dim,
                dim // 2
            ),
            nn.Tanh(),
            nn.Linear(
                dim // 2,
                1
            )
        )

    def forward(
        self,
        x,
        mask=None
    ):

        scores = self.score(x)

        if mask is not None:

            scores = scores.float()

            scores = scores.masked_fill(
                ~mask.unsqueeze(-1),
                -1e4
            )

        weights = torch.softmax(
            scores,
            dim=1
        )

        return torch.sum(
            x * weights,
            dim=1
        )

In [30]:
# ============================================================
# CELL 24 — IR SEQUENCE ENCODER
# ============================================================

class IRSequenceEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = IRFrameEncoder()

        self.lstm = nn.LSTM(
            input_size=384,
            hidden_size=256,
            num_layers=2,
            batch_first=True,
            dropout=0.25,
            bidirectional=True
        )

        self.attention = TemporalAttention(
            dim=512
        )

        self.embedding_size = 512

    def forward(
        self,
        x,
        mask=None
    ):
        """
        x:
            [B,T,3,H,W]

        mask:
            [B,T]
        """

        B, T, C, H, W = x.shape

        x = x.reshape(
            B * T,
            C,
            H,
            W
        )

        x = self.encoder(x)

        x = x.reshape(
            B,
            T,
            384
        )

        x, _ = self.lstm(x)

        x = self.attention(
            x,
            mask
        )

        return x

# ============================================================
# CELL 25 — LOAD IR ENCODER
# ============================================================

ir_encoder = IRSequenceEncoder().to(DEVICE)

missing, unexpected = ir_encoder.load_state_dict(
    ir_state,
    strict=True
)

print("IR encoder loaded ✓")
print("Missing:", missing)
print("Unexpected:", unexpected)
print("Embedding size:", ir_encoder.embedding_size)

ir_encoder.eval()

IR encoder loaded ✓
Missing: []
Unexpected: []
Embedding size: 512


IRSequenceEncoder(
  (encoder): IRFrameEncoder(
    (stem): Sequential(
      (0): Conv2d(3, 32, kernel_size=(5, 5), stride=(2, 2), padding=(2, 2), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): GELU(approximate='none')
    )
    (layer1): Sequential(
      (0): ResidualBlock(
        (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (shortcut): Identity()
      )
      (1): ResidualBlock(
        (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Con

In [31]:
# ============================================================
# CELL 26 — DEPTH + COLOR ENCODER
# ============================================================

class ConvBlock(nn.Module):

    def __init__(
        self,
        in_ch,
        out_ch,
        stride=1
    ):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_ch,
            out_ch,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(out_ch)

        self.conv2 = nn.Conv2d(
            out_ch,
            out_ch,
            kernel_size=3,
            padding=1,
            bias=False
        )

        self.bn2 = nn.BatchNorm2d(out_ch)

        if stride != 1 or in_ch != out_ch:

            self.skip = nn.Sequential(
                nn.Conv2d(
                    in_ch,
                    out_ch,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),
                nn.BatchNorm2d(out_ch)
            )

        else:

            self.skip = nn.Identity()

    def forward(self, x):

        residual = self.skip(x)

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.silu(x)

        x = self.conv2(x)
        x = self.bn2(x)

        x = x + residual

        x = F.silu(x)

        return x


class DepthSpatialEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(
            ConvBlock(3, 32, stride=2),
            ConvBlock(32, 64, stride=2),
            ConvBlock(64, 96, stride=2),
            ConvBlock(96, 128, stride=2),
        )

        self.pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):

        x = self.net(x)

        x = self.pool(x)

        return x.flatten(1)


class DepthTemporalBlock(nn.Module):

    def __init__(
        self,
        channels,
        dilation,
        dropout=0.15
    ):
        super().__init__()

        self.conv1 = nn.Conv1d(
            channels,
            channels,
            kernel_size=3,
            padding=dilation,
            dilation=dilation
        )

        self.bn1 = nn.BatchNorm1d(channels)

        self.conv2 = nn.Conv1d(
            channels,
            channels,
            kernel_size=3,
            padding=dilation,
            dilation=dilation
        )

        self.bn2 = nn.BatchNorm1d(channels)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        residual = x

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.silu(x)
        x = self.dropout(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.dropout(x)

        x = x + residual

        x = F.silu(x)

        return x


class DepthTemporalAttention(nn.Module):

    def __init__(self, dim):

        super().__init__()

        self.score = nn.Sequential(
            nn.Linear(
                dim,
                dim // 2
            ),
            nn.Tanh(),
            nn.Linear(
                dim // 2,
                1
            )
        )

    def forward(
        self,
        x,
        mask=None
    ):

        scores = self.score(x)

        if mask is not None:

            scores = scores.float()

            scores = scores.masked_fill(
                ~mask.unsqueeze(-1),
                -1e4
            )

        weights = torch.softmax(
            scores,
            dim=1
        )

        return torch.sum(
            x * weights,
            dim=1
        )


class DepthColorEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.cnn = DepthSpatialEncoder()

        self.frame_proj = nn.Sequential(
            nn.Linear(128, 128),
            nn.LayerNorm(128),
            nn.SiLU(),
            nn.Dropout(0.10)
        )

        self.temporal = nn.Sequential(
            DepthTemporalBlock(128, dilation=1),
            DepthTemporalBlock(128, dilation=2),
            DepthTemporalBlock(128, dilation=4),
            DepthTemporalBlock(128, dilation=8),
        )

        self.attention = DepthTemporalAttention(128)

        self.embedding_size = 128

    def forward(
        self,
        x,
        mask=None
    ):
        """
        x: [B,T,3,H,W]
        mask: [B,T]
        """

        B, T, C, H, W = x.shape

        x = x.reshape(
            B * T,
            C,
            H,
            W
        )

        x = self.cnn(x)

        x = self.frame_proj(x)

        x = x.reshape(
            B,
            T,
            128
        )

        # TCN expects [B,C,T]
        x = x.transpose(1, 2)

        x = self.temporal(x)

        x = x.transpose(1, 2)

        x = self.attention(
            x,
            mask
        )

        return x

In [35]:
# ============================================================
# CELL 27 — LOAD DEPTH ENCODER
# ============================================================

depth_encoder = DepthColorEncoder().to(DEVICE)

# Remove the original 40-class classifier.
depth_encoder_state = {
    k: v
    for k, v in depth_state.items()
    if not k.startswith("classifier.")
}

missing, unexpected = depth_encoder.load_state_dict(
    depth_encoder_state,
    strict=True
)

print("Depth encoder loaded ✓")
print("Missing:", missing)
print("Unexpected:", unexpected)
print("Embedding size:", depth_encoder.embedding_size)

Depth encoder loaded ✓
Missing: []
Unexpected: []
Embedding size: 128


In [36]:
# ============================================================
# CELL 28 — RADAR ENCODER
# ============================================================

class DetectionEncoder(nn.Module):

    def __init__(
        self,
        input_dim=6,
        hidden_dim=64,
        output_dim=128
    ):
        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),

            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),

            nn.Linear(hidden_dim, output_dim),
            nn.GELU()
        )

    def forward(self, x):
        """
        x: [B,T,D,6]
        """

        B, T, D, Fdim = x.shape

        flat = x.reshape(
            B * T * D,
            Fdim
        )

        encoded = self.mlp(flat)

        encoded = encoded.reshape(
            B,
            T,
            D,
            128
        )

        # Detection exists when original
        # feature vector is non-zero.
        detection_mask = (
            x.abs().sum(dim=-1) > 0
        )

        mask = detection_mask.unsqueeze(-1)

        # Mean pooling
        masked = encoded * mask

        count = mask.sum(
            dim=2
        ).clamp_min(1)

        mean_pool = (
            masked.sum(dim=2)
            / count
        )

        # Max pooling
        neg_inf = torch.finfo(
            encoded.dtype
        ).min

        max_input = encoded.masked_fill(
            ~mask,
            neg_inf
        )

        max_pool = max_input.max(
            dim=2
        ).values

        # For completely empty frames,
        # max becomes invalid; replace it.
        empty = (
            ~detection_mask.any(dim=2)
        )

        if empty.any():
            max_pool = max_pool.masked_fill(
                empty.unsqueeze(-1),
                0
            )

        return torch.cat(
            [
                mean_pool,
                max_pool
            ],
            dim=-1
        )


class RadarTCNBlock(nn.Module):

    def __init__(
        self,
        channels=256,
        dilation=1,
        dropout=0.15
    ):
        super().__init__()

        self.conv1 = nn.Conv1d(
            channels,
            channels,
            kernel_size=3,
            padding=dilation,
            dilation=dilation
        )

        self.bn1 = nn.BatchNorm1d(channels)

        self.conv2 = nn.Conv1d(
            channels,
            channels,
            kernel_size=3,
            padding=dilation,
            dilation=dilation
        )

        self.bn2 = nn.BatchNorm1d(channels)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        residual = x

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.gelu(x)
        x = self.dropout(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = F.gelu(x)
        x = self.dropout(x)

        x = x + residual

        x = F.gelu(x)

        return x


class RadarAttentionPooling(nn.Module):

    def __init__(self):

        super().__init__()

        self.score = nn.Sequential(
            nn.Linear(256, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

    def forward(
        self,
        x,
        mask=None
    ):

        scores = self.score(x)

        if mask is not None:

            scores = scores.float()

            scores = scores.masked_fill(
                ~mask.unsqueeze(-1),
                torch.finfo(scores.dtype).min
            )

        weights = torch.softmax(
            scores,
            dim=1
        )

        return torch.sum(
            x * weights,
            dim=1
        )


class RadarEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.point_encoder = DetectionEncoder(
            input_dim=6,
            hidden_dim=64,
            output_dim=128
        )

        self.frame_projection = nn.Sequential(
            nn.Linear(256, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.15)
        )

        self.tcn = nn.Sequential(
            RadarTCNBlock(256, dilation=1),
            RadarTCNBlock(256, dilation=2),
            RadarTCNBlock(256, dilation=4),
            RadarTCNBlock(256, dilation=8),
        )

        self.attention = RadarAttentionPooling()

        self.embedding_size = 256

    def forward(
        self,
        x,
        frame_mask=None
    ):
        """
        x:
            [B,T,D,6]

        frame_mask:
            [B,T]
        """

        x = self.point_encoder(x)

        x = self.frame_projection(x)

        # [B,T,256] → [B,256,T]
        x = x.transpose(1, 2)

        x = self.tcn(x)

        x = x.transpose(1, 2)

        x = self.attention(
            x,
            frame_mask
        )

        return x

# ============================================================
# CELL 29 — LOAD RADAR ENCODER
# ============================================================

radar_encoder = RadarEncoder().to(DEVICE)

# Remove the original 40-class classifier.
radar_encoder_state = {
    k: v
    for k, v in radar_state.items()
    if not k.startswith("classifier.")
}

missing, unexpected = radar_encoder.load_state_dict(
    radar_encoder_state,
    strict=True
)

print("Radar encoder loaded ✓")
print("Missing:", missing)
print("Unexpected:", unexpected)
print("Embedding size:", radar_encoder.embedding_size)

Radar encoder loaded ✓
Missing: []
Unexpected: []
Embedding size: 256


---
---

In [38]:
# ============================================================
# CELL 32 — SKELETON DATASET
# ============================================================

from pathlib import Path
import json
import numpy as np
import torch
from torch.utils.data import Dataset


class FusionSkeletonDataset(Dataset):

    def __init__(
        self,
        df,
        sequence_length=64
    ):
        self.df = df.reset_index(drop=True)
        self.sequence_length = sequence_length

        self.parents = [
            -1, 0, 0, 1, 2,
            11, 12, 5, 6,
            7, 8, -1, -1,
            11, 12, 13, 14
        ]

    def load_skeleton(self, path):

        path = Path(path)

        json_files = sorted(
            (path / "predictions").glob("*.json")
        )

        keypoints = []
        scores = []

        for json_file in json_files:

            try:
                with open(
                    json_file,
                    "r"
                ) as f:
                    data = json.load(f)

                if not data:
                    continue

                item = data[0]

                kp = np.asarray(
                    item["keypoints"],
                    dtype=np.float32
                )

                sc = np.asarray(
                    item["keypoint_scores"],
                    dtype=np.float32
                )

                if kp.shape != (17, 3):
                    continue

                if sc.shape != (17,):
                    continue

                kp = np.nan_to_num(
                    kp,
                    nan=0.0,
                    posinf=0.0,
                    neginf=0.0
                )

                sc = np.nan_to_num(
                    sc,
                    nan=0.0,
                    posinf=0.0,
                    neginf=0.0
                )

                sc = np.clip(
                    sc,
                    0.0,
                    1.0
                )

                keypoints.append(kp)
                scores.append(sc)

            except Exception:
                continue

        if len(keypoints) == 0:
            return None, None

        return (
            np.stack(keypoints),
            np.stack(scores)
        )

    def normalize_skeleton(self, x):

        # Pelvis = mean of left/right hip
        pelvis = (
            x[:, 11] +
            x[:, 12]
        ) / 2.0

        x = x - pelvis[:, None, :]

        # Shoulder center
        shoulder_center = (
            x[:, 5] +
            x[:, 6]
        ) / 2.0

        scale = np.linalg.norm(
            shoulder_center,
            axis=-1,
            keepdims=True
        )

        scale = np.maximum(
            scale,
            1e-6
        )

        x = x / scale[:, None, :]

        return x

    def resample(self, x):

        T = len(x)

        if T == self.sequence_length:

            return x

        if T == 1:

            return np.repeat(
                x,
                self.sequence_length,
                axis=0
            )

        old_idx = np.arange(T)

        new_idx = np.linspace(
            0,
            T - 1,
            self.sequence_length
        )

        out = np.empty(
            (
                self.sequence_length,
                x.shape[1],
                x.shape[2]
            ),
            dtype=np.float32
        )

        for j in range(x.shape[1]):

            for f in range(x.shape[2]):

                out[:, j, f] = np.interp(
                    new_idx,
                    old_idx,
                    x[:, j, f]
                )

        return out.astype(np.float32)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        skeleton, scores = self.load_skeleton(
            row["skeleton_path"]
        )

        if skeleton is None:

            # Fallback preserving expected shape.
            skeleton = np.zeros(
                (self.sequence_length, 17, 3),
                dtype=np.float32
            )

        else:

            skeleton = self.normalize_skeleton(
                skeleton
            )

            velocity = np.diff(
                skeleton,
                axis=0,
                prepend=skeleton[0:1]
            )

            acceleration = np.diff(
                velocity,
                axis=0,
                prepend=velocity[0:1]
            )

            bones = np.zeros_like(
                skeleton
            )

            for j, parent in enumerate(
                self.parents
            ):

                if parent >= 0:

                    bones[:, j] = (
                        skeleton[:, j]
                        - skeleton[:, parent]
                    )

            features = np.concatenate(
                [
                    skeleton,
                    velocity,
                    acceleration,
                    bones
                ],
                axis=-1
            )

            skeleton = self.resample(
                features
            )

        x = torch.from_numpy(
            skeleton
        ).float()

        y = torch.tensor(
            int(row["class_id"]),
            dtype=torch.long
        )

        return x, y

# ============================================================
# CELL 33 — REAL SKELETON FORWARD TEST
# ============================================================

test_skel_ds = FusionSkeletonDataset(
    fusion_train_df.iloc[:4],
    sequence_length=64
)

x, y = test_skel_ds[0]

print("Input shape:", tuple(x.shape))
print("Label:", y.item())

assert x.shape == (
    64,
    17,
    12
)

with torch.no_grad():

    z = skeleton_encoder(
        x.unsqueeze(0).to(DEVICE)
    )

print(
    "Embedding shape:",
    tuple(z.shape)
)

assert z.shape == (1, 256)

print("✓ REAL SKELETON FORWARD PASS SUCCESSFUL")

Input shape: (64, 17, 12)
Label: 0
Embedding shape: (1, 256)
✓ REAL SKELETON FORWARD PASS SUCCESSFUL


In [39]:
# ============================================================
# CELL 34 — IR DATASET
# ============================================================

import cv2
from PIL import Image


class FusionIRDataset(Dataset):

    def __init__(
        self,
        df,
        seq_len=24,
        img_w=160,
        img_h=120,
        train=False
    ):
        self.df = df.reset_index(drop=True)
        self.seq_len = seq_len
        self.img_w = img_w
        self.img_h = img_h
        self.train = train

    def _frame_number(self, p):

        try:
            return int(
                p.stem.split("_")[-1]
            )
        except Exception:
            return -1

    def _normalize(self, img):

        img = img.astype(
            np.float32
        )

        p2 = np.percentile(
            img,
            2
        )

        p98 = np.percentile(
            img,
            98
        )

        if p98 > p2:

            img = (
                img - p2
            ) / (
                p98 - p2
            )

            img = np.clip(
                img,
                0.0,
                1.0
            )

        else:

            img = img / 255.0

        return img.astype(
            np.float32
        )

    def _select_frames(self, files):

        n = len(files)

        if n == 0:
            return []

        if n >= self.seq_len:

            if self.train:

                max_start = n - self.seq_len

                if max_start > 0:

                    start = np.random.randint(
                        0,
                        max_start + 1
                    )

                else:

                    start = 0

                indices = np.linspace(
                    start,
                    n - 1,
                    self.seq_len
                ).astype(np.int64)

            else:

                indices = np.linspace(
                    0,
                    n - 1,
                    self.seq_len
                ).astype(np.int64)

        else:

            indices = np.linspace(
                0,
                n - 1,
                self.seq_len
            ).astype(np.int64)

        return [
            files[i]
            for i in indices
        ]

    def _process_frame(self, path):

        try:

            raw = cv2.imread(
                str(path),
                cv2.IMREAD_GRAYSCALE
            )

            if raw is None:
                raise ValueError(
                    "Could not read image"
                )

            raw = self._normalize(
                raw
            )

            raw = cv2.resize(
                raw,
                (
                    self.img_w,
                    self.img_h
                ),
                interpolation=cv2.INTER_AREA
            )

            gx = cv2.Sobel(
                raw,
                cv2.CV_32F,
                1,
                0,
                ksize=3
            )

            gy = cv2.Sobel(
                raw,
                cv2.CV_32F,
                0,
                1,
                ksize=3
            )

            magnitude = np.sqrt(
                gx * gx +
                gy * gy
            )

            magnitude = (
                magnitude /
                (
                    magnitude.max()
                    + 1e-6
                )
            )

            return raw, magnitude

        except Exception:

            return None, None

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        trial_path = Path(
            row["ir_path"]
        )

        files = sorted(
            trial_path.glob("*.png"),
            key=self._frame_number
        )

        selected = self._select_frames(
            files
        )

        frames = []
        mask = []

        previous_raw = None

        for path in selected:

            try:

                raw = cv2.imread(
                    str(path),
                    cv2.IMREAD_GRAYSCALE
                )

                if raw is None:
                    raise ValueError()

                raw = self._normalize(
                    raw
                )

                raw = cv2.resize(
                    raw,
                    (
                        self.img_w,
                        self.img_h
                    ),
                    interpolation=cv2.INTER_AREA
                )

                gx = cv2.Sobel(
                    raw,
                    cv2.CV_32F,
                    1,
                    0,
                    ksize=3
                )

                gy = cv2.Sobel(
                    raw,
                    cv2.CV_32F,
                    0,
                    1,
                    ksize=3
                )

                magnitude = np.sqrt(
                    gx * gx +
                    gy * gy
                )

                magnitude /= (
                    magnitude.max()
                    + 1e-6
                )

                if previous_raw is None:

                    difference = np.zeros_like(
                        raw
                    )

                else:

                    difference = np.abs(
                        raw -
                        previous_raw
                    )

                previous_raw = raw.copy()

                frame = np.stack(
                    [
                        raw,
                        magnitude,
                        difference
                    ],
                    axis=0
                )

                frames.append(frame)
                mask.append(True)

            except Exception:

                frames.append(
                    np.zeros(
                        (
                            3,
                            self.img_h,
                            self.img_w
                        ),
                        dtype=np.float32
                    )
                )

                mask.append(False)

        # Safety padding
        while len(frames) < self.seq_len:

            frames.append(
                np.zeros(
                    (
                        3,
                        self.img_h,
                        self.img_w
                    ),
                    dtype=np.float32
                )
            )

            mask.append(False)

        frames = frames[:self.seq_len]
        mask = mask[:self.seq_len]

        x = np.stack(
            frames
        ).astype(np.float32)

        # ----------------------------------------------------
        # Training augmentation
        # ----------------------------------------------------

        if self.train:

            if np.random.rand() < 0.5:

                x = x[:, :, :, ::-1].copy()

            if np.random.rand() < 0.3:

                factor = np.random.uniform(
                    0.90,
                    1.10
                )

                x[:, 0] = np.clip(
                    x[:, 0] * factor,
                    0.0,
                    1.0
                )

        x = torch.from_numpy(
            x
        ).float()

        mask = torch.tensor(
            mask,
            dtype=torch.bool
        )

        y = torch.tensor(
            int(row["class_id"]),
            dtype=torch.long
        )

        return x, mask, y

# ============================================================
# CELL 35 — REAL IR FORWARD TEST
# ============================================================

test_ir_ds = FusionIRDataset(
    fusion_train_df.iloc[:4],
    seq_len=24,
    img_w=160,
    img_h=120,
    train=False
)

x, mask, y = test_ir_ds[0]

print("Input shape:", tuple(x.shape))
print("Mask shape:", tuple(mask.shape))
print("Valid frames:", mask.sum().item())
print("Label:", y.item())

assert x.shape == (
    24,
    3,
    120,
    160
)

assert mask.shape == (24,)

with torch.no_grad():

    z = ir_encoder(
        x.unsqueeze(0).to(DEVICE),
        mask.unsqueeze(0).to(DEVICE)
    )

print(
    "Embedding shape:",
    tuple(z.shape)
)

assert z.shape == (1, 512)

print("✓ REAL IR FORWARD PASS SUCCESSFUL")

Input shape: (24, 3, 120, 160)
Mask shape: (24,)
Valid frames: 24
Label: 0
Embedding shape: (1, 512)
✓ REAL IR FORWARD PASS SUCCESSFUL


In [40]:
# ============================================================
# CELL 36 — DEPTH + COLOR DATASET
# ============================================================

from PIL import Image


class FusionDepthDataset(Dataset):

    def __init__(
        self,
        df,
        max_frames=32,
        img_size=160,
        train=False
    ):
        self.df = df.reset_index(drop=True)
        self.max_frames = max_frames
        self.img_size = img_size
        self.train = train

        self.mean = np.array(
            [0.485, 0.456, 0.406],
            dtype=np.float32
        )

        self.std = np.array(
            [0.229, 0.224, 0.225],
            dtype=np.float32
        )

    def select_frames(self, files):

        n = len(files)

        if n <= self.max_frames:

            return files

        if self.train:

            max_start = (
                n - self.max_frames
            )

            start = np.random.randint(
                0,
                max_start + 1
            )

            indices = np.arange(
                start,
                start + self.max_frames
            )

        else:

            indices = np.linspace(
                0,
                n - 1,
                self.max_frames
            ).astype(np.int64)

        return [
            files[i]
            for i in indices
        ]

    def transform_frame(self, path):

        try:

            img = Image.open(
                path
            ).convert("RGB")

            # ------------------------------------------------
            # Training augmentation
            # ------------------------------------------------

            if self.train:

                if np.random.rand() < 0.5:

                    img = img.transpose(
                        Image.Transpose.FLIP_LEFT_RIGHT
                    )

                if np.random.rand() < 0.35:

                    w, h = img.size

                    scale = np.random.uniform(
                        0.88,
                        1.0
                    )

                    crop_w = max(
                        1,
                        int(w * scale)
                    )

                    crop_h = max(
                        1,
                        int(h * scale)
                    )

                    if (
                        crop_w < w
                        or crop_h < h
                    ):

                        left = np.random.randint(
                            0,
                            w - crop_w + 1
                        )

                        top = np.random.randint(
                            0,
                            h - crop_h + 1
                        )

                        img = img.crop(
                            (
                                left,
                                top,
                                left + crop_w,
                                top + crop_h
                            )
                        )

            img = img.resize(
                (
                    self.img_size,
                    self.img_size
                ),
                Image.Resampling.BILINEAR
            )

            arr = np.asarray(
                img,
                dtype=np.float32
            ) / 255.0

            arr = (
                arr - self.mean
            ) / self.std

            arr = np.transpose(
                arr,
                (2, 0, 1)
            )

            return arr.astype(
                np.float32
            ), True

        except Exception:

            return (
                np.zeros(
                    (
                        3,
                        self.img_size,
                        self.img_size
                    ),
                    dtype=np.float32
                ),
                False
            )

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        files = row["files"]

        # Handle paths that may have been
        # serialized as strings.
        if isinstance(files, str):

            try:
                files = json.loads(files)
            except Exception:
                files = [
                    files
                ]

        files = [
            Path(f)
            for f in files
        ]

        selected = self.select_frames(
            files
        )

        frames = []
        mask = []

        for path in selected:

            frame, valid = (
                self.transform_frame(path)
            )

            frames.append(frame)
            mask.append(valid)

        # ----------------------------------------------------
        # Pad to MAX_FRAMES
        # ----------------------------------------------------

        while len(frames) < self.max_frames:

            frames.append(
                np.zeros(
                    (
                        3,
                        self.img_size,
                        self.img_size
                    ),
                    dtype=np.float32
                )
            )

            mask.append(False)

        frames = frames[:self.max_frames]
        mask = mask[:self.max_frames]

        x = np.stack(
            frames
        ).astype(np.float32)

        mask = np.asarray(
            mask,
            dtype=bool
        )

        y = int(
            row["class_id"]
        )

        return (
            torch.from_numpy(x).float(),
            torch.from_numpy(mask),
            torch.tensor(
                y,
                dtype=torch.long
            )
        )

# ============================================================
# CELL 37 — REAL DEPTH FORWARD TEST
# ============================================================

test_depth_ds = FusionDepthDataset(
    fusion_train_df.iloc[:4],
    max_frames=32,
    img_size=160,
    train=False
)

x, mask, y = test_depth_ds[0]

print("Input shape:", tuple(x.shape))
print("Mask shape:", tuple(mask.shape))
print("Valid frames:", mask.sum().item())
print("Label:", y.item())

assert x.shape == (
    32,
    3,
    160,
    160
)

assert mask.shape == (
    32,
)

with torch.no_grad():

    z = depth_encoder(
        x.unsqueeze(0).to(DEVICE),
        mask.unsqueeze(0).to(DEVICE)
    )

print(
    "Embedding shape:",
    tuple(z.shape)
)

assert z.shape == (1, 128)

print(
    "✓ REAL DEPTH FORWARD PASS SUCCESSFUL"
)

Input shape: (32, 3, 160, 160)
Mask shape: (32,)
Valid frames: 20
Label: 0
Embedding shape: (1, 128)
✓ REAL DEPTH FORWARD PASS SUCCESSFUL


---

In [73]:
# ============================================================
# CELL 50B — RADAR FEATURE ENCODER
# ============================================================

class RadarFeatureEncoder(nn.Module):

    def __init__(self, radar_model):
        super().__init__()

        self.point_encoder = radar_model.point_encoder
        self.frame_projection = radar_model.frame_projection
        self.tcn = radar_model.tcn
        self.attention = radar_model.attention

        self.embedding_size = 256

    def forward(self, x, frame_mask):

        B, T, D, Fdim = x.shape

        # ----------------------------------------------------
        # Detection mask
        # ----------------------------------------------------

        detection_mask = (
            x.abs().sum(dim=-1) > 0
        )

        # ----------------------------------------------------
        # Point / detection encoding
        # ----------------------------------------------------

        frames = self.point_encoder(
            x,
            detection_mask
        )

        # [B,T,256]
        frames = self.frame_projection(
            frames
        )

        # [B,T,C] → [B,C,T]
        frames = frames.transpose(
            1,
            2
        )

        # ----------------------------------------------------
        # Temporal TCN
        # ----------------------------------------------------

        frames = self.tcn(
            frames
        )

        # [B,C,T] → [B,T,C]
        frames = frames.transpose(
            1,
            2
        )

        # ----------------------------------------------------
        # Temporal attention
        # ----------------------------------------------------

        pooled = self.attention(
            frames,
            frame_mask
        )

        # [B,256]
        return pooled


radar_feature_encoder = RadarFeatureEncoder(
    new_radar_encoder_model
).to(DEVICE)

radar_feature_encoder.eval()

print(
    "Radar embedding size:",
    radar_feature_encoder.embedding_size
)
# ============================================================
# CELL 50C — TEST 256-D RADAR EMBEDDING
# ============================================================

x, mask, y = radar_real_ds[0]

with torch.no_grad():

    z = radar_feature_encoder(
        x.unsqueeze(0).to(DEVICE),
        mask.unsqueeze(0).to(DEVICE)
    )

print("Input shape:", tuple(x.shape))
print("Valid frames:", mask.sum().item())
print(
    "Non-zero detections:",
    (x.abs().sum(dim=-1) > 0).sum().item()
)

print(
    "Radar embedding shape:",
    tuple(z.shape)
)

assert z.shape == (1, 256)

print(
    "\n✓ NEW 22% RADAR 256-D FEATURE EXTRACTION SUCCESSFUL"
)

Radar embedding size: 256
Input shape: (128, 32, 6)
Valid frames: 82
Non-zero detections: 267
Radar embedding shape: (1, 256)

✓ NEW 22% RADAR 256-D FEATURE EXTRACTION SUCCESSFUL


In [71]:
# ============================================================
# CELL 50A — VERIFY RADAR ENCODER LOAD
# ============================================================

expected_classifier_keys = [
    "classifier.0.weight",
    "classifier.0.bias",
    "classifier.1.weight",
    "classifier.1.bias",
    "classifier.4.weight",
    "classifier.4.bias"
]

print("Missing keys:")
print(missing)

print("\nUnexpected keys:")
print(unexpected)

# These are intentionally missing because we removed
# the 40-class classifier for feature extraction.
assert set(missing) == set(
    expected_classifier_keys
)

assert len(unexpected) == 0

new_radar_encoder_model.eval()

print(
    "\n✓ RADAR ENCODER WEIGHTS LOADED CORRECTLY"
)

# ------------------------------------------------------------
# REAL RADAR FORWARD TEST
# ------------------------------------------------------------

x, mask, y = radar_real_ds[0]

print("\nReal Radar sample:")
print(
    "Input shape:",
    tuple(x.shape)
)

print(
    "Mask shape:",
    tuple(mask.shape)
)

print(
    "Valid frames:",
    mask.sum().item()
)

print(
    "Non-zero detections:",
    (
        x.abs().sum(dim=-1) > 0
    ).sum().item()
)

print(
    "Label:",
    y.item()
)

assert mask.sum().item() > 0

with torch.no_grad():

    z = new_radar_encoder_model(
        x.unsqueeze(0).to(DEVICE),
        mask.unsqueeze(0).to(DEVICE)
    )

print(
    "Embedding shape:",
    tuple(z.shape)
)

assert z.shape == (1, 256)

print(
    "\n✓ NEW RADAR REAL FORWARD PASS SUCCESSFUL"
)

Missing keys:
['classifier.0.weight', 'classifier.0.bias', 'classifier.1.weight', 'classifier.1.bias', 'classifier.4.weight', 'classifier.4.bias']

Unexpected keys:
[]

✓ RADAR ENCODER WEIGHTS LOADED CORRECTLY

Real Radar sample:
Input shape: (128, 32, 6)
Mask shape: (128,)
Valid frames: 82
Non-zero detections: 267
Label: 0
Embedding shape: (1, 40)


AssertionError: 

In [46]:
# ============================================================
# CELL 43 — FIND A GENUINELY USABLE RADAR RECORDING
# ============================================================

valid_radar_rows = []

for i, row in radar_available_df.iterrows():

    path = Path(row["radar_path"])

    try:
        df_r = pd.read_csv(path)

        if len(df_r) > 0:
            valid_radar_rows.append(i)

            if len(valid_radar_rows) >= 10:
                break

    except Exception:
        continue


print(
    "Usable Radar recordings found:",
    len(valid_radar_rows)
)

assert len(valid_radar_rows) > 0

real_radar_df = radar_available_df.iloc[
    valid_radar_rows
].reset_index(drop=True)

print("\nFirst usable recording:")
print(real_radar_df.iloc[0][
    [
        "user",
        "trial",
        "class_id",
        "radar_path"
    ]
].to_string())

Usable Radar recordings found: 10

First usable recording:
user                                                      user4
trial                                                     1-1-1
class_id                                                      0
radar_path    /kaggle/input/datasets/samasiayushman/small-mo...


In [47]:
# ============================================================
# CELL 44 — GENUINE RADAR FORWARD PASS
# ============================================================

radar_real_ds = FusionRadarDataset(
    real_radar_df.iloc[:10],
    max_frames=128,
    max_detections=32,
    train=False,
    feature_mean=RADAR_MEAN,
    feature_std=RADAR_STD
)

x, mask, y = radar_real_ds[0]

print("Input shape:", tuple(x.shape))
print("Mask shape:", tuple(mask.shape))
print("Valid frames:", mask.sum().item())
print(
    "Non-zero detections:",
    (x.abs().sum(dim=-1) > 0).sum().item()
)
print("Label:", y.item())

assert x.shape == (128, 32, 6)
assert mask.shape == (128,)

assert mask.sum().item() > 0

with torch.no_grad():

    z = radar_encoder(
        x.unsqueeze(0).to(DEVICE),
        mask.unsqueeze(0).to(DEVICE)
    )

print(
    "Embedding shape:",
    tuple(z.shape)
)

assert z.shape == (1, 256)

print(
    "✓ GENUINE RADAR FORWARD PASS SUCCESSFUL"
)

Input shape: (128, 32, 6)
Mask shape: (128,)
Valid frames: 82
Non-zero detections: 267
Label: 0
Embedding shape: (1, 256)
✓ GENUINE RADAR FORWARD PASS SUCCESSFUL


---

In [49]:
# ============================================================
# CELL 41 — FUSION EMBEDDING CACHE SETUP
# ============================================================

CACHE_ROOT = Path(
    "/kaggle/working/cuhk_x_fusion_cache"
)

CACHE_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

for name in [
    "skeleton",
    "ir",
    "depth",
    "radar"
]:
    (CACHE_ROOT / name).mkdir(
        parents=True,
        exist_ok=True
    )

print("Cache root:", CACHE_ROOT)

print("\nFusion splits:")
print("Train:", len(fusion_train_df))
print("Val  :", len(fusion_val_df))

print("\nTrain users:")
print(
    sorted(
        fusion_train_df["user"].unique()
    )
)

print("\nVal users:")
print(
    sorted(
        fusion_val_df["user"].unique()
    )
)

# ============================================================
# CELL 42 — SKELETON CACHE DATASETS / LOADERS
# ============================================================

class FusionSkeletonDataset(Dataset):

    def __init__(
        self,
        df,
        sequence_length=64
    ):
        self.df = df.reset_index(drop=True)
        self.sequence_length = sequence_length

    def __len__(self):
        return len(self.df)

    def load_skeleton(self, path):

        pred_dir = Path(path) / "predictions"

        json_files = sorted(
            pred_dir.glob("*.json")
        )

        keypoints = []
        scores = []

            # ------------------------------------------------
            # Read valid frames
            # ------------------------------------------------

        for jf in json_files:

            try:

                with open(
                    jf,
                    "r"
                ) as f:
                    data = json.load(f)

                if not data:
                    continue

                person = data[0]

                kp = np.asarray(
                    person["keypoints"],
                    dtype=np.float32
                )

                sc = np.asarray(
                    person["keypoint_scores"],
                    dtype=np.float32
                )

                if kp.shape != (17, 3):
                    continue

                if sc.shape != (17,):
                    continue

                kp = np.nan_to_num(
                    kp,
                    nan=0.0,
                    posinf=0.0,
                    neginf=0.0
                )

                sc = np.clip(
                    np.nan_to_num(
                        sc,
                        nan=0.0,
                        posinf=0.0,
                        neginf=0.0
                    ),
                    0.0,
                    1.0
                )

                keypoints.append(kp)
                scores.append(sc)

            except Exception:
                continue

        if len(keypoints) == 0:
            return None, None

        return (
            np.stack(keypoints).astype(
                np.float32
            ),
            np.stack(scores).astype(
                np.float32
            )
        )

    def normalize_skeleton(self, x):

        # ----------------------------------------------------
        # Pelvis-centered normalization
        # pelvis = joints 11 and 12
        # ----------------------------------------------------

        pelvis = (
            x[:, 11] +
            x[:, 12]
        ) / 2.0

        x = x - pelvis[:, None, :]

        # ----------------------------------------------------
        # Shoulder-center scale
        # shoulders = joints 5 and 6
        # ----------------------------------------------------

        shoulder_center = (
            x[:, 5] +
            x[:, 6]
        ) / 2.0

        scale = np.linalg.norm(
            shoulder_center,
            axis=1,
            keepdims=True
        )

        scale = np.maximum(
            scale,
            1e-6
        )

        x = x / scale[:, :, None]

        return x.astype(
            np.float32
        )

    def resample(self, arr):

        T = len(arr)

        if T == self.sequence_length:
            return arr

        if T == 1:

            return np.repeat(
                arr,
                self.sequence_length,
                axis=0
            )

        old_idx = np.linspace(
            0,
            1,
            T
        )

        new_idx = np.linspace(
            0,
            1,
            self.sequence_length
        )

        out = np.empty(
            (
                self.sequence_length,
                *arr.shape[1:]
            ),
            dtype=np.float32
        )

        for j in range(
            arr.shape[1]
        ):

            for k in range(
                arr.shape[2]
            ):

                out[:, j, k] = np.interp(
                    new_idx,
                    old_idx,
                    arr[:, j, k]
                )

        return out

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        skeleton, scores = (
            self.load_skeleton(
                row["skeleton_path"]
            )
        )

        if skeleton is None:

            skeleton = np.zeros(
                (1, 17, 3),
                dtype=np.float32
            )

        skeleton = self.normalize_skeleton(
            skeleton
        )

        # ----------------------------------------------------
        # Velocity
        # ----------------------------------------------------

        velocity = np.diff(
            skeleton,
            axis=0,
            prepend=skeleton[:1]
        )

        # ----------------------------------------------------
        # Acceleration
        # ----------------------------------------------------

        acceleration = np.diff(
            velocity,
            axis=0,
            prepend=velocity[:1]
        )

        # ----------------------------------------------------
        # COCO-17 bone vectors
        # ----------------------------------------------------

        parents = [
            -1, 0, 0, 1, 2,
            11, 12, 5, 6,
            7, 8, -1, -1,
            11, 12, 13, 14
        ]

        bones = np.zeros_like(
            skeleton
        )

        for j, p in enumerate(
            parents
        ):

            if p >= 0:
                bones[:, j] = (
                    skeleton[:, j]
                    - skeleton[:, p]
                )

        # ----------------------------------------------------
        # IMPORTANT:
        # The verified original pipeline uses ONLY
        #
        # skeleton + velocity + acceleration + bones
        #
        # = 12 features per joint.
        #
        # Confidence is NOT concatenated.
        # ----------------------------------------------------

        features = np.concatenate(
            [
                skeleton,
                velocity,
                acceleration,
                bones
            ],
            axis=-1
        )

        features = self.resample(
            features
        )

        assert features.shape == (
            64,
            17,
            12
        )

        return (
            torch.from_numpy(
                features
            ).float(),

            torch.tensor(
                int(row["class_id"]),
                dtype=torch.long
            )
        )


skeleton_cache_train_ds = FusionSkeletonDataset(
    fusion_train_df
)

skeleton_cache_val_ds = FusionSkeletonDataset(
    fusion_val_df
)

skeleton_cache_train_loader = DataLoader(
    skeleton_cache_train_ds,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

skeleton_cache_val_loader = DataLoader(
    skeleton_cache_val_ds,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(
    "Skeleton train batches:",
    len(skeleton_cache_train_loader)
)

print(
    "Skeleton val batches:",
    len(skeleton_cache_val_loader)
)
# ============================================================
# CELL 43 — CACHE SKELETON EMBEDDINGS
# ============================================================

def cache_skeleton_embeddings(
    loader,
    encoder,
    split_name
):

    encoder.eval()

    embeddings = []
    labels = []

    with torch.no_grad():

        for batch_idx, (
            x,
            y
        ) in enumerate(loader):

            x = x.to(
                DEVICE,
                non_blocking=True
            )

            z = encoder(x)

            embeddings.append(
                z.cpu()
            )

            labels.append(
                y.cpu()
            )

            if (
                batch_idx + 1
            ) % 20 == 0:

                print(
                    f"{split_name}: "
                    f"{batch_idx + 1}/"
                    f"{len(loader)}"
                )

    embeddings = torch.cat(
        embeddings,
        dim=0
    )

    labels = torch.cat(
        labels,
        dim=0
    )

    print(
        f"\n{split_name} embeddings:",
        tuple(embeddings.shape)
    )

    print(
        f"{split_name} labels:",
        tuple(labels.shape)
    )

    return embeddings, labels


skel_train_emb, skel_train_y = (
    cache_skeleton_embeddings(
        skeleton_cache_train_loader,
        skeleton_encoder,
        "Skeleton Train"
    )
)

skel_val_emb, skel_val_y = (
    cache_skeleton_embeddings(
        skeleton_cache_val_loader,
        skeleton_encoder,
        "Skeleton Val"
    )
)

assert skel_train_emb.shape == (
    len(fusion_train_df),
    256
)

assert skel_val_emb.shape == (
    len(fusion_val_df),
    256
)

print(
    "\n✓ SKELETON EMBEDDINGS CACHED"
)

Cache root: /kaggle/working/cuhk_x_fusion_cache

Fusion splits:
Train: 2238
Val  : 693

Train users:
['user17', 'user18', 'user19', 'user20', 'user21', 'user23', 'user24', 'user3', 'user4', 'user5', 'user6', 'user7', 'user8', 'user9']

Val users:
['user1', 'user16', 'user2', 'user22']
Skeleton train batches: 70
Skeleton val batches: 22
Skeleton Train: 20/70
Skeleton Train: 40/70
Skeleton Train: 60/70

Skeleton Train embeddings: (2238, 256)
Skeleton Train labels: (2238,)
Skeleton Val: 20/22

Skeleton Val embeddings: (693, 256)
Skeleton Val labels: (693,)

✓ SKELETON EMBEDDINGS CACHED


In [52]:
# ============================================================
# CELL 44 — IR CACHE DATASETS / LOADERS
# ============================================================

# ============================================================
# CELL 44A — FIX IR DATASET LENGTH
# ============================================================

def _ir_dataset_len(self):
    return len(self.df)

FusionIRDataset.__len__ = _ir_dataset_len

print(
    "✓ FusionIRDataset.__len__ added"
)

print(
    "Dataset length:",
    len(FusionIRDataset(
        fusion_train_df,
        seq_len=24,
        img_w=160,
        img_h=120,
        train=False
    ))
)

ir_cache_train_ds = FusionIRDataset(
    fusion_train_df,
    seq_len=24,
    img_w=160,
    img_h=120,
    train=False
)

ir_cache_val_ds = FusionIRDataset(
    fusion_val_df,
    seq_len=24,
    img_w=160,
    img_h=120,
    train=False
)

ir_cache_train_loader = DataLoader(
    ir_cache_train_ds,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

ir_cache_val_loader = DataLoader(
    ir_cache_val_ds,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(
    "IR train samples:",
    len(ir_cache_train_ds)
)

print(
    "IR val samples:",
    len(ir_cache_val_ds)
)

print(
    "IR train batches:",
    len(ir_cache_train_loader)
)

print(
    "IR val batches:",
    len(ir_cache_val_loader)
)
# ============================================================
# CELL 45 — CACHE IR EMBEDDINGS
# ============================================================

def cache_ir_embeddings(
    loader,
    encoder,
    split_name
):

    encoder.eval()

    embeddings = []
    labels = []

    with torch.no_grad():

        for batch_idx, (
            x,
            mask,
            y
        ) in enumerate(loader):

            x = x.to(
                DEVICE,
                non_blocking=True
            )

            mask = mask.to(
                DEVICE,
                non_blocking=True
            )

            z = encoder(
                x,
                mask
            )

            embeddings.append(
                z.cpu()
            )

            labels.append(
                y.cpu()
            )

            if (
                batch_idx + 1
            ) % 20 == 0:

                print(
                    f"{split_name}: "
                    f"{batch_idx + 1}/"
                    f"{len(loader)}"
                )

    embeddings = torch.cat(
        embeddings,
        dim=0
    )

    labels = torch.cat(
        labels,
        dim=0
    )

    print(
        f"\n{split_name} embeddings:",
        tuple(embeddings.shape)
    )

    print(
        f"{split_name} labels:",
        tuple(labels.shape)
    )

    return embeddings, labels


ir_train_emb, ir_train_y = (
    cache_ir_embeddings(
        ir_cache_train_loader,
        ir_encoder,
        "IR Train"
    )
)

ir_val_emb, ir_val_y = (
    cache_ir_embeddings(
        ir_cache_val_loader,
        ir_encoder,
        "IR Val"
    )
)

assert ir_train_emb.shape == (
    len(fusion_train_df),
    512
)

assert ir_val_emb.shape == (
    len(fusion_val_df),
    512
)

print(
    "\n✓ IR EMBEDDINGS CACHED"
)

✓ FusionIRDataset.__len__ added
Dataset length: 2238
IR train samples: 2238
IR val samples: 693
IR train batches: 140
IR val batches: 44
IR Train: 20/140
IR Train: 40/140
IR Train: 60/140
IR Train: 80/140
IR Train: 100/140
IR Train: 120/140
IR Train: 140/140

IR Train embeddings: (2238, 512)
IR Train labels: (2238,)
IR Val: 20/44
IR Val: 40/44

IR Val embeddings: (693, 512)
IR Val labels: (693,)

✓ IR EMBEDDINGS CACHED


In [54]:
# ============================================================
# CELL 46 — DEPTH CACHE DATASETS / LOADERS
# ============================================================
# ============================================================
# CELL 46A — FIX DEPTH DATASET LENGTH
# ============================================================

def _depth_dataset_len(self):
    return len(self.df)

FusionDepthDataset.__len__ = _depth_dataset_len

print("✓ FusionDepthDataset.__len__ added")

print(
    "Dataset length:",
    len(
        FusionDepthDataset(
            fusion_train_df,
            max_frames=32,
            img_size=160,
            train=False
        )
    )
)

depth_cache_train_ds = FusionDepthDataset(
    fusion_train_df,
    max_frames=32,
    img_size=160,
    train=False
)

depth_cache_val_ds = FusionDepthDataset(
    fusion_val_df,
    max_frames=32,
    img_size=160,
    train=False
)

depth_cache_train_loader = DataLoader(
    depth_cache_train_ds,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

depth_cache_val_loader = DataLoader(
    depth_cache_val_ds,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(
    "Depth train samples:",
    len(depth_cache_train_ds)
)

print(
    "Depth val samples:",
    len(depth_cache_val_ds)
)

print(
    "Depth train batches:",
    len(depth_cache_train_loader)
)

print(
    "Depth val batches:",
    len(depth_cache_val_loader)
)

# ============================================================
# CELL 47 — CACHE DEPTH EMBEDDINGS
# ============================================================

def cache_depth_embeddings(
    loader,
    encoder,
    split_name
):

    encoder.eval()

    embeddings = []
    labels = []

    with torch.no_grad():

        for batch_idx, (
            x,
            mask,
            y
        ) in enumerate(loader):

            x = x.to(
                DEVICE,
                non_blocking=True
            )

            mask = mask.to(
                DEVICE,
                non_blocking=True
            )

            z = encoder(
                x,
                mask
            )

            embeddings.append(
                z.cpu()
            )

            labels.append(
                y.cpu()
            )

            if (
                batch_idx + 1
            ) % 20 == 0:

                print(
                    f"{split_name}: "
                    f"{batch_idx + 1}/"
                    f"{len(loader)}"
                )

    embeddings = torch.cat(
        embeddings,
        dim=0
    )

    labels = torch.cat(
        labels,
        dim=0
    )

    print(
        f"\n{split_name} embeddings:",
        tuple(embeddings.shape)
    )

    print(
        f"{split_name} labels:",
        tuple(labels.shape)
    )

    return embeddings, labels


depth_train_emb, depth_train_y = (
    cache_depth_embeddings(
        depth_cache_train_loader,
        depth_encoder,
        "Depth Train"
    )
)

depth_val_emb, depth_val_y = (
    cache_depth_embeddings(
        depth_cache_val_loader,
        depth_encoder,
        "Depth Val"
    )
)

assert depth_train_emb.shape == (
    len(fusion_train_df),
    128
)

assert depth_val_emb.shape == (
    len(fusion_val_df),
    128
)

print(
    "\n✓ DEPTH EMBEDDINGS CACHED"
)

✓ FusionDepthDataset.__len__ added
Dataset length: 2238
Depth train samples: 2238
Depth val samples: 693
Depth train batches: 280
Depth val batches: 87
Depth Train: 20/280
Depth Train: 40/280
Depth Train: 60/280
Depth Train: 80/280
Depth Train: 100/280
Depth Train: 120/280
Depth Train: 140/280
Depth Train: 160/280
Depth Train: 180/280
Depth Train: 200/280
Depth Train: 220/280
Depth Train: 240/280
Depth Train: 260/280
Depth Train: 280/280

Depth Train embeddings: (2238, 128)
Depth Train labels: (2238,)
Depth Val: 20/87
Depth Val: 40/87
Depth Val: 60/87
Depth Val: 80/87

Depth Val embeddings: (693, 128)
Depth Val labels: (693,)

✓ DEPTH EMBEDDINGS CACHED


In [81]:
# ============================================================
# CELL 52 — RADAR CACHE DATASET SETUP
# ============================================================
# ============================================================
# CELL 52A — CORRECTED RADAR DATASET FOR FULL FUSION INDEX
# ============================================================

class FusionRadarDataset(Dataset):

    def __init__(
        self,
        df,
        max_frames=128,
        max_detections=32,
        train=False,
        feature_mean=None,
        feature_std=None
    ):

        self.df = df.reset_index(drop=True)

        self.max_frames = max_frames
        self.max_detections = max_detections
        self.train = train

        self.feature_mean = np.asarray(
            feature_mean,
            dtype=np.float32
        )

        self.feature_std = np.asarray(
            feature_std,
            dtype=np.float32
        )

    def __len__(self):
        return len(self.df)

    def _empty_sample(self):

        sequence = np.zeros(
            (
                self.max_frames,
                self.max_detections,
                6
            ),
            dtype=np.float32
        )

        frame_mask = np.zeros(
            self.max_frames,
            dtype=bool
        )

        return sequence, frame_mask

    def _load_csv(self, path):

        try:

            df = pd.read_csv(path)

            required = [
                "frame",
                "x",
                "y",
                "z",
                "v",
                "snr",
                "noise"
            ]

            if not all(
                c in df.columns
                for c in required
            ):
                return None

            # ------------------------------------------------
            # Numeric conversion
            # ------------------------------------------------

            for c in required:

                df[c] = pd.to_numeric(
                    df[c],
                    errors="coerce"
                )

            df = df.replace(
                [np.inf, -np.inf],
                np.nan
            )

            df = df.dropna(
                subset=required
            )

            if len(df) == 0:
                return None

            return df

        except Exception:

            return None

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        # ====================================================
        # IMPORTANT:
        # Missing Radar path = valid fusion sample,
        # but Radar modality is unavailable.
        # ====================================================

        radar_path = row["radar_path"]

        if (
            pd.isna(radar_path)
            or not isinstance(
                radar_path,
                (str, os.PathLike)
            )
        ):

            sequence, frame_mask = (
                self._empty_sample()
            )

            return (
                torch.from_numpy(
                    sequence
                ).float(),

                torch.from_numpy(
                    frame_mask
                ),

                torch.tensor(
                    int(row["class_id"]),
                    dtype=torch.long
                )
            )

        path = Path(
            radar_path
        )

        # ----------------------------------------------------
        # Path doesn't actually exist
        # ----------------------------------------------------

        if not path.exists():

            sequence, frame_mask = (
                self._empty_sample()
            )

            return (
                torch.from_numpy(
                    sequence
                ).float(),

                torch.from_numpy(
                    frame_mask
                ),

                torch.tensor(
                    int(row["class_id"]),
                    dtype=torch.long
                )
            )

        df = self._load_csv(
            path
        )

        # ----------------------------------------------------
        # Empty / unusable CSV
        # ----------------------------------------------------

        if df is None:

            sequence, frame_mask = (
                self._empty_sample()
            )

            return (
                torch.from_numpy(
                    sequence
                ).float(),

                torch.from_numpy(
                    frame_mask
                ),

                torch.tensor(
                    int(row["class_id"]),
                    dtype=torch.long
                )
            )

        # ====================================================
        # Frame IDs
        # ====================================================

        frame_ids = sorted(
            df["frame"].unique()
        )

        # ----------------------------------------------------
        # Temporal sampling
        # ----------------------------------------------------

        if len(frame_ids) > self.max_frames:

            if self.train:

                start = np.random.randint(
                    0,
                    len(frame_ids)
                    - self.max_frames
                    + 1
                )

            else:

                start = (
                    len(frame_ids)
                    - self.max_frames
                ) // 2

            frame_ids = frame_ids[
                start:
                start + self.max_frames
            ]

        # ====================================================
        # Fixed-size sequence
        # ====================================================

        sequence = np.zeros(
            (
                self.max_frames,
                self.max_detections,
                6
            ),
            dtype=np.float32
        )

        frame_mask = np.zeros(
            self.max_frames,
            dtype=bool
        )

        # ====================================================
        # Construct frames
        # ====================================================

        for t, frame_id in enumerate(
            frame_ids
        ):

            if t >= self.max_frames:
                break

            frame_df = df[
                df["frame"] == frame_id
            ]

            values = frame_df[
                [
                    "x",
                    "y",
                    "z",
                    "v",
                    "snr",
                    "noise"
                ]
            ].values.astype(
                np.float32
            )

            if len(values) == 0:
                continue

            # ------------------------------------------------
            # Strongest detections by SNR
            # ------------------------------------------------

            if len(values) > self.max_detections:

                order = np.argsort(
                    values[:, 4]
                )[::-1]

                values = values[
                    order[
                        :self.max_detections
                    ]
                ]

            # ------------------------------------------------
            # Exact checkpoint normalization
            # ------------------------------------------------

            values = (
                values
                - self.feature_mean
            ) / self.feature_std

            n = min(
                len(values),
                self.max_detections
            )

            sequence[
                t,
                :n
            ] = values[:n]

            frame_mask[t] = True

        # ====================================================
        # Training augmentation
        # ====================================================

        if self.train:

            if np.random.rand() < 0.5:

                noise = np.random.normal(
                    0,
                    0.015,
                    size=sequence.shape
                ).astype(
                    np.float32
                )

                sequence += (
                    noise
                    * frame_mask[
                        :, None, None
                    ]
                )

            if np.random.rand() < 0.30:

                for t in range(
                    self.max_frames
                ):

                    if not frame_mask[t]:
                        continue

                    keep = (
                        np.random.rand(
                            self.max_detections
                        ) > 0.08
                    )

                    sequence[
                        t,
                        ~keep
                    ] = 0

        return (
            torch.from_numpy(
                sequence
            ).float(),

            torch.from_numpy(
                frame_mask
            ),

            torch.tensor(
                int(row["class_id"]),
                dtype=torch.long
            )
        )
# Make sure FusionRadarDataset has __len__
def _radar_dataset_len(self):
    return len(self.df)

FusionRadarDataset.__len__ = _radar_dataset_len


# ------------------------------------------------------------
# IMPORTANT:
# Use the NEW 22.57% checkpoint statistics.
# They should match the values we already verified.
# ------------------------------------------------------------

RADAR_MEAN = np.asarray(
    radar_checkpoint["feature_mean"],
    dtype=np.float32
)

RADAR_STD = np.asarray(
    radar_checkpoint["feature_std"],
    dtype=np.float32
)

print("Radar mean:")
print(RADAR_MEAN)

print("\nRadar std:")
print(RADAR_STD)


# ------------------------------------------------------------
# Full fusion datasets
#
# train=False:
# deterministic center-window sampling
# ------------------------------------------------------------


# ============================================================
# CELL 52B — RECREATE RADAR CACHE LOADERS
# ============================================================

radar_cache_train_ds = FusionRadarDataset(
    fusion_train_df,
    max_frames=128,
    max_detections=32,
    train=False,
    feature_mean=RADAR_MEAN,
    feature_std=RADAR_STD
)

radar_cache_val_ds = FusionRadarDataset(
    fusion_val_df,
    max_frames=128,
    max_detections=32,
    train=False,
    feature_mean=RADAR_MEAN,
    feature_std=RADAR_STD
)

radar_cache_train_loader = DataLoader(
    radar_cache_train_ds,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

radar_cache_val_loader = DataLoader(
    radar_cache_val_ds,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(
    "Radar train:",
    len(radar_cache_train_ds)
)

print(
    "Radar val:",
    len(radar_cache_val_ds)
)

print(
    "Train batches:",
    len(radar_cache_train_loader)
)

print(
    "Val batches:",
    len(radar_cache_val_loader)
)

Radar mean:
[-1.7317031e-01  1.0887016e+00 -1.6865708e-01 -7.1119308e-04
  1.7384203e+02  5.2611737e+02]

Radar std:
[ 0.88422245  1.1882622   1.2405075   0.15830402 45.527454   65.53308   ]
Radar train: 2238
Radar val: 693
Train batches: 140
Val batches: 44


In [83]:
# ============================================================
# CELL 52C — TEST AVAILABLE + MISSING RADAR
# ============================================================

# ------------------------------------------------------------
# Find one missing Radar trial
# ------------------------------------------------------------

missing_indices = np.where(
    fusion_train_df["radar_path"].isna()
)[0]

print(
    "Missing Radar paths:",
    len(missing_indices)
)

if len(missing_indices) > 0:

    idx = missing_indices[0]

    x_missing, mask_missing, y_missing = (
        radar_cache_train_ds[idx]
    )

    print("\nMissing Radar test:")
    print(
        "Input:",
        tuple(x_missing.shape)
    )

    print(
        "Valid frames:",
        mask_missing.sum().item()
    )

    print(
        "Non-zero detections:",
        (
            x_missing.abs().sum(dim=-1) > 0
        ).sum().item()
    )

    assert mask_missing.sum().item() == 0


# ------------------------------------------------------------
# Find one Radar path that actually has data
# ------------------------------------------------------------

available_indices = []

for i in range(
    min(len(fusion_train_df), 500)
):

    rp = fusion_train_df.iloc[i][
        "radar_path"
    ]

    if pd.isna(rp):
        continue

    try:

        if Path(rp).stat().st_size > 0:

            temp = pd.read_csv(rp)

            if len(temp) > 0:
                available_indices.append(i)
                break

    except Exception:
        continue


assert len(available_indices) > 0

idx = available_indices[0]

x_real, mask_real, y_real = (
    radar_cache_train_ds[idx]
)

print("\nAvailable Radar test:")
print(
    "Input:",
    tuple(x_real.shape)
)

print(
    "Valid frames:",
    mask_real.sum().item()
)

print(
    "Non-zero detections:",
    (
        x_real.abs().sum(dim=-1) > 0
    ).sum().item()
)

assert mask_real.sum().item() > 0

print(
    "\n✓ MISSING + AVAILABLE RADAR HANDLING VERIFIED"
)

Missing Radar paths: 10

Missing Radar test:
Input: (128, 32, 6)
Valid frames: 0
Non-zero detections: 0

Available Radar test:
Input: (128, 32, 6)
Valid frames: 82
Non-zero detections: 267

✓ MISSING + AVAILABLE RADAR HANDLING VERIFIED


In [86]:
# ============================================================
# CELL 56A — CORRECT MULTIMODAL ALIGNMENT CHECK
# ============================================================

# ------------------------------------------------------------
# 1. Check dataframe columns
# ------------------------------------------------------------

print("Fusion columns:")
print(fusion_train_df.columns.tolist())

print("\nTrain rows:", len(fusion_train_df))
print("Val rows:", len(fusion_val_df))


# ------------------------------------------------------------
# 2. Verify every cached label matches the dataframe
# ------------------------------------------------------------

train_labels = torch.tensor(
    fusion_train_df["class_id"].values,
    dtype=torch.long
)

val_labels = torch.tensor(
    fusion_val_df["class_id"].values,
    dtype=torch.long
)


for name, y in [
    ("Skeleton", skel_train_y),
    ("IR", ir_train_y),
    ("Depth", depth_train_y),
    ("Radar", radar_train_y),
]:

    assert torch.equal(
        y,
        train_labels
    )

    print(
        f"✓ {name} train labels aligned"
    )


for name, y in [
    ("Skeleton", skel_val_y),
    ("IR", ir_val_y),
    ("Depth", depth_val_y),
    ("Radar", radar_val_y),
]:

    assert torch.equal(
        y,
        val_labels
    )

    print(
        f"✓ {name} val labels aligned"
    )


# ------------------------------------------------------------
# 3. Verify the actual trial identity columns
# ------------------------------------------------------------

IDENTITY_COLS = [
    "user",
    "trial",
    "class_id"
]

print(
    "\nChecking identity columns:",
    IDENTITY_COLS
)

for col in IDENTITY_COLS:

    assert col in fusion_train_df.columns
    assert col in fusion_val_df.columns


# ------------------------------------------------------------
# 4. Check duplicate identities
# ------------------------------------------------------------

train_dupes = fusion_train_df.duplicated(
    subset=IDENTITY_COLS
).sum()

val_dupes = fusion_val_df.duplicated(
    subset=IDENTITY_COLS
).sum()

print(
    "\nTrain duplicate identities:",
    train_dupes
)

print(
    "Val duplicate identities:",
    val_dupes
)

assert train_dupes == 0
assert val_dupes == 0


# ------------------------------------------------------------
# 5. Check train/validation user separation
# ------------------------------------------------------------

train_users = set(
    fusion_train_df["user"]
)

val_users = set(
    fusion_val_df["user"]
)

print(
    "\nTrain users:",
    sorted(train_users)
)

print(
    "Val users:",
    sorted(val_users)
)

assert len(
    train_users & val_users
) == 0

print(
    "\n✓ NO USER LEAKAGE"
)


# ------------------------------------------------------------
# 6. Radar availability
# ------------------------------------------------------------

print("\nRadar availability:")

print(
    "Train:",
    int(radar_train_present.sum()),
    "/",
    len(radar_train_present)
)

print(
    "Val:",
    int(radar_val_present.sum()),
    "/",
    len(radar_val_present)
)


# ------------------------------------------------------------
# FINAL
# ------------------------------------------------------------

print(
    "\n" + "=" * 65
)

print(
    "✓ COMPLETE MULTIMODAL CACHE ALIGNMENT PASSED"
)

print("=" * 65)

Fusion columns:
['action', 'class_id', 'user', 'trial', 'skeleton_path', 'depth_path', 'files', 'n_files', 'radar_path', 'ir_n_files_x', 'ir_path', 'ir_n_files_y', 'has_skeleton', 'has_ir', 'has_depth', 'has_radar']

Train rows: 2238
Val rows: 693
✓ Skeleton train labels aligned
✓ IR train labels aligned
✓ Depth train labels aligned
✓ Radar train labels aligned
✓ Skeleton val labels aligned
✓ IR val labels aligned
✓ Depth val labels aligned
✓ Radar val labels aligned

Checking identity columns: ['user', 'trial', 'class_id']

Train duplicate identities: 0
Val duplicate identities: 0

Train users: ['user17', 'user18', 'user19', 'user20', 'user21', 'user23', 'user24', 'user3', 'user4', 'user5', 'user6', 'user7', 'user8', 'user9']
Val users: ['user1', 'user16', 'user2', 'user22']

✓ NO USER LEAKAGE

Radar availability:
Train: 1110 / 2238
Val: 299 / 693

✓ COMPLETE MULTIMODAL CACHE ALIGNMENT PASSED


In [87]:
# ============================================================
# CELL 58 — SAVE COMPLETE MULTIMODAL EMBEDDING CACHE
# ============================================================

from pathlib import Path
import torch
import json

CACHE_DIR = Path("/kaggle/working/cuhk_x_fusion_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Save training cache
# ------------------------------------------------------------

train_cache = {
    "skeleton": skel_train_emb.cpu(),
    "ir": ir_train_emb.cpu(),
    "depth": depth_train_emb.cpu(),
    "radar": radar_train_emb.cpu(),

    "labels": skel_train_y.cpu(),

    "radar_present": radar_train_present.cpu(),

    "users": fusion_train_df["user"].astype(str).tolist(),
    "trials": fusion_train_df["trial"].astype(str).tolist(),
    "actions": fusion_train_df["action"].astype(str).tolist(),
}

torch.save(
    train_cache,
    CACHE_DIR / "train_embeddings.pt"
)


# ------------------------------------------------------------
# Save validation cache
# ------------------------------------------------------------

val_cache = {
    "skeleton": skel_val_emb.cpu(),
    "ir": ir_val_emb.cpu(),
    "depth": depth_val_emb.cpu(),
    "radar": radar_val_emb.cpu(),

    "labels": skel_val_y.cpu(),

    "radar_present": radar_val_present.cpu(),

    "users": fusion_val_df["user"].astype(str).tolist(),
    "trials": fusion_val_df["trial"].astype(str).tolist(),
    "actions": fusion_val_df["action"].astype(str).tolist(),
}

torch.save(
    val_cache,
    CACHE_DIR / "val_embeddings.pt"
)


# ------------------------------------------------------------
# Save master index
# ------------------------------------------------------------

fusion_train_df.to_csv(
    CACHE_DIR / "fusion_train_index.csv",
    index=False
)

fusion_val_df.to_csv(
    CACHE_DIR / "fusion_val_index.csv",
    index=False
)


# ------------------------------------------------------------
# Save cache metadata
# ------------------------------------------------------------

metadata = {
    "train_samples": len(train_cache["labels"]),
    "val_samples": len(val_cache["labels"]),

    "num_classes": 40,

    "embedding_dimensions": {
        "skeleton": 256,
        "ir": 512,
        "depth": 128,
        "radar": 256,
    },

    "radar_train_present": int(
        train_cache["radar_present"].sum()
    ),

    "radar_val_present": int(
        val_cache["radar_present"].sum()
    ),
}

with open(
    CACHE_DIR / "metadata.json",
    "w"
) as f:
    json.dump(metadata, f, indent=2)


print("=" * 65)
print("✓ FUSION CACHE SAVED")
print("=" * 65)

for p in CACHE_DIR.iterdir():
    print(
        f"{p.name:30s} "
        f"{p.stat().st_size / 1024**2:.2f} MB"
    )

✓ FUSION CACHE SAVED
val_embeddings.pt              3.07 MB
train_embeddings.pt            9.91 MB
skeleton                       0.00 MB
fusion_train_index.csv         11.73 MB
radar                          0.00 MB
fusion_val_index.csv           3.93 MB
depth                          0.00 MB
ir                             0.00 MB
fusion_master_index.csv        15.66 MB
metadata.json                  0.00 MB


In [88]:
# ============================================================
# CELL 59 — RELOAD CACHE VERIFICATION
# ============================================================

train_reload = torch.load(
    CACHE_DIR / "train_embeddings.pt",
    map_location="cpu"
)

val_reload = torch.load(
    CACHE_DIR / "val_embeddings.pt",
    map_location="cpu"
)


# ------------------------------------------------------------
# Shapes
# ------------------------------------------------------------

print("TRAIN")

for key in [
    "skeleton",
    "ir",
    "depth",
    "radar",
    "labels",
    "radar_present"
]:
    print(
        f"{key:15s}",
        tuple(train_reload[key].shape)
    )


print("\nVAL")

for key in [
    "skeleton",
    "ir",
    "depth",
    "radar",
    "labels",
    "radar_present"
]:
    print(
        f"{key:15s}",
        tuple(val_reload[key].shape)
    )


# ------------------------------------------------------------
# Exact tensor equality
# ------------------------------------------------------------

assert torch.equal(
    train_reload["skeleton"],
    skel_train_emb.cpu()
)

assert torch.equal(
    train_reload["ir"],
    ir_train_emb.cpu()
)

assert torch.equal(
    train_reload["depth"],
    depth_train_emb.cpu()
)

assert torch.equal(
    train_reload["radar"],
    radar_train_emb.cpu()
)

assert torch.equal(
    train_reload["labels"],
    skel_train_y.cpu()
)

assert torch.equal(
    train_reload["radar_present"],
    radar_train_present.cpu()
)


assert torch.equal(
    val_reload["skeleton"],
    skel_val_emb.cpu()
)

assert torch.equal(
    val_reload["ir"],
    ir_val_emb.cpu()
)

assert torch.equal(
    val_reload["depth"],
    depth_val_emb.cpu()
)

assert torch.equal(
    val_reload["radar"],
    radar_val_emb.cpu()
)

assert torch.equal(
    val_reload["labels"],
    skel_val_y.cpu()
)

assert torch.equal(
    val_reload["radar_present"],
    radar_val_present.cpu()
)


print()
print("=" * 65)
print("✓ CACHE RELOAD VERIFIED EXACTLY")
print("=" * 65)

TRAIN
skeleton        (2238, 256)
ir              (2238, 512)
depth           (2238, 128)
radar           (2238, 256)
labels          (2238,)
radar_present   (2238,)

VAL
skeleton        (693, 256)
ir              (693, 512)
depth           (693, 128)
radar           (693, 256)
labels          (693,)
radar_present   (693,)

✓ CACHE RELOAD VERIFIED EXACTLY


# CELL 60 — BIG FEATURE-LEVEL MULTIMODAL FUSION MODEL


In [ ]:
# ============================================================
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

MODALITIES = ["skeleton", "ir", "depth"]

In [173]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# MODALITY PROJECTOR
# ============================================================

class StrongProjector(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim=512,
        dropout=0.15
    ):
        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(input_dim, hidden_dim),

            nn.LayerNorm(hidden_dim),
            nn.GELU(),

            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim),

            nn.LayerNorm(hidden_dim)
        )

    def forward(self, x):
        return self.net(x)


# ============================================================
# CROSS MODAL TRANSFORMER BLOCK
# ============================================================

class CrossModalBlock(nn.Module):

    def __init__(
        self,
        hidden_dim=512,
        heads=8,
        dropout=0.20
    ):
        super().__init__()

        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm1 = nn.LayerNorm(hidden_dim)

        self.ffn = nn.Sequential(

            nn.Linear(
                hidden_dim,
                hidden_dim * 2
            ),

            nn.GELU(),

            nn.Dropout(dropout),

            nn.Linear(
                hidden_dim * 2,
                hidden_dim
            ),

            nn.Dropout(dropout)
        )

        self.norm2 = nn.LayerNorm(hidden_dim)

    def forward(self, x):

        attended, _ = self.attention(
            x,
            x,
            x
        )

        x = self.norm1(
            x + attended
        )

        x = self.norm2(
            x + self.ffn(x)
        )

        return x


# ============================================================
# MODALITY GATE
# ============================================================

class StrongModalityGate(nn.Module):

    def __init__(
        self,
        hidden_dim=512
    ):
        super().__init__()

        self.score = nn.Sequential(

            nn.Linear(
                hidden_dim,
                128
            ),

            nn.GELU(),

            nn.Dropout(0.15),

            nn.Linear(
                128,
                1
            )
        )

    def forward(self, tokens):

        scores = self.score(
            tokens
        ).squeeze(-1)

        weights = torch.softmax(
            scores,
            dim=1
        )

        gated = (
            tokens *
            weights.unsqueeze(-1)
        )

        return gated, weights


# ============================================================
# SERIOUS S + IR FUSION
# ============================================================

class SeriousSIRFusion(nn.Module):

    def __init__(
        self,
        skeleton_dim=256,
        ir_dim=512,
        hidden_dim=512,
        num_classes=40,
        num_cross_blocks=3,
        heads=8,
        dropout=0.20
    ):
        super().__init__()

        # ----------------------------------------------------
        # Projectors
        # ----------------------------------------------------

        self.skeleton_proj = StrongProjector(
            skeleton_dim,
            hidden_dim,
            dropout=0.15
        )

        self.ir_proj = StrongProjector(
            ir_dim,
            hidden_dim,
            dropout=0.15
        )

        # ----------------------------------------------------
        # Modality identity
        # ----------------------------------------------------

        self.modality_embedding = nn.Parameter(
            torch.randn(
                1,
                2,
                hidden_dim
            ) * 0.02
        )

        # ----------------------------------------------------
        # Cross-modal processing
        # ----------------------------------------------------

        self.cross_blocks = nn.ModuleList([

            CrossModalBlock(
                hidden_dim=hidden_dim,
                heads=heads,
                dropout=dropout
            )

            for _ in range(num_cross_blocks)
        ])

        # ----------------------------------------------------
        # Gate
        # ----------------------------------------------------

        self.gate = StrongModalityGate(
            hidden_dim
        )

        # ----------------------------------------------------
        # Explicit interactions
        #
        # S
        # IR
        # S × IR
        # |S - IR|
        # S + IR
        #
        # = 5 × 512
        # ----------------------------------------------------

        interaction_dim = hidden_dim * 5

        self.interaction = nn.Sequential(

            nn.Linear(
                interaction_dim,
                1024
            ),

            nn.LayerNorm(1024),

            nn.GELU(),

            nn.Dropout(0.25),

            nn.Linear(
                1024,
                512
            ),

            nn.LayerNorm(512),

            nn.GELU(),

            nn.Dropout(0.20)
        )

        # ----------------------------------------------------
        # Global fusion
        # ----------------------------------------------------

        self.global_fusion = nn.Sequential(

            nn.Linear(
                hidden_dim * 2,
                768
            ),

            nn.LayerNorm(768),

            nn.GELU(),

            nn.Dropout(0.25),

            nn.Linear(
                768,
                512
            ),

            nn.LayerNorm(512),

            nn.GELU(),

            nn.Dropout(0.20)
        )

        # ----------------------------------------------------
        # Final fusion
        # ----------------------------------------------------

        self.final_fusion = nn.Sequential(

            nn.Linear(
                512 + 512,
                768
            ),

            nn.LayerNorm(768),

            nn.GELU(),

            nn.Dropout(0.30),

            nn.Linear(
                768,
                512
            ),

            nn.LayerNorm(512),

            nn.GELU(),

            nn.Dropout(0.20)
        )

        # ----------------------------------------------------
        # Residual
        # ----------------------------------------------------

        self.residual = nn.Sequential(

            nn.Linear(
                512 + 512,
                512
            ),

            nn.LayerNorm(512)
        )

        # ----------------------------------------------------
        # Classifier
        # ----------------------------------------------------

        self.classifier = nn.Sequential(

            nn.Linear(
                512,
                384
            ),

            nn.LayerNorm(384),

            nn.GELU(),

            nn.Dropout(0.30),

            nn.Linear(
                384,
                192
            ),

            nn.LayerNorm(192),

            nn.GELU(),

            nn.Dropout(0.20),

            nn.Linear(
                192,
                num_classes
            )
        )

    def forward(
        self,
        skeleton,
        ir
    ):

        # ----------------------------------------------------
        # Project
        # ----------------------------------------------------

        s = self.skeleton_proj(
            skeleton
        )

        i = self.ir_proj(
            ir
        )

        # ----------------------------------------------------
        # Tokens
        # ----------------------------------------------------

        tokens = torch.stack(
            [s, i],
            dim=1
        )

        tokens = (
            tokens +
            self.modality_embedding
        )

        # ----------------------------------------------------
        # Cross-modal transformer
        # ----------------------------------------------------

        for block in self.cross_blocks:

            tokens = block(
                tokens
            )

        # ----------------------------------------------------
        # Extract
        # ----------------------------------------------------

        s = tokens[:, 0]
        i = tokens[:, 1]

        # ----------------------------------------------------
        # Gating
        # ----------------------------------------------------

        gated_tokens, gate_weights = (
            self.gate(tokens)
        )

        gated = gated_tokens.flatten(1)

        # ----------------------------------------------------
        # Explicit interactions
        # ----------------------------------------------------

        interaction_input = torch.cat(
            [
                s,
                i,
                s * i,
                torch.abs(s - i),
                s + i
            ],
            dim=-1
        )

        interaction_features = (
            self.interaction(
                interaction_input
            )
        )

        # ----------------------------------------------------
        # Global
        # ----------------------------------------------------

        global_features = (
            self.global_fusion(
                torch.cat(
                    [s, i],
                    dim=-1
                )
            )
        )

        # ----------------------------------------------------
        # Combine
        #
        # gated = 1024
        # interaction = 512
        # global = 512
        #
        # total = 2048
        # ----------------------------------------------------

        combined = torch.cat(
            [
                gated,
                interaction_features,
                global_features
            ],
            dim=-1
        )

        # Compress
        fusion_input = torch.cat(
            [
                interaction_features,
                global_features
            ],
            dim=-1
        )

        fused = (
            self.final_fusion(
                fusion_input
            )
            +
            self.residual(
                fusion_input
            )
        )

        

        # ----------------------------------------------------
        # Classification
        # ----------------------------------------------------

        logits = self.classifier(
            fused
        )

        return logits, {
            "gate_weights": gate_weights,
            "interaction_features":
                interaction_features,
            "global_features":
                global_features,
            "fused_features":
                fused
        }

In [174]:
# ============================================================
# CELL 61 — MODEL SIZE + STRUCTURAL CHECK
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model =  SeriousSIRFusion(
    skeleton_dim=256,
    ir_dim=512,
    hidden_dim=512,
    num_classes=40,
    num_cross_blocks=3,
    heads=8,
    dropout=0.20
).to(device)



# ------------------------------------------------------------
# Parameter count
# ------------------------------------------------------------

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

fp32_mb = (
    total_params * 4
) / (1024 ** 2)

print("=" * 65)
print("BIG MULTIMODAL FUSION MODEL")
print("=" * 65)

print(f"Device:          {device}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable:        {trainable_params:,}")
print(f"FP32 parameter:   {fp32_mb:.2f} MB")

BIG MULTIMODAL FUSION MODEL
Device:          cuda
Total parameters: 13,621,737
Trainable:        13,621,737
FP32 parameter:   51.96 MB


In [145]:
# ============================================================
# CELL 62 — REAL DATA FORWARD TEST
# ============================================================

model.eval()

B = 8

s = train_reload["skeleton"][:B].to(device)
i = train_reload["ir"][:B].to(device)
d = train_reload["depth"][:B].to(device)
rp = train_reload["radar_present"][:B].to(device)

with torch.no_grad():

    logits, aux = model(
        skeleton=s,
        ir=i,
        depth=d,
        radar=r,
        radar_present=rp
    )


print("=" * 65)
print("FORWARD TEST")
print("=" * 65)

print("Skeleton:", s.shape)
print("IR:      ", i.shape)
print("Depth:   ", d.shape)
print("Radar:   ", r.shape)
print("Radar mask:", rp.shape)

print()
print("Logits:", logits.shape)
print("Gate:", aux["gate_weights"].shape)
print("Pair:", aux["pair_features"].shape)
print("Global:", aux["global_features"].shape)
print("Fused:", aux["fused_features"].shape)


assert logits.shape == (B, 40)
assert aux["gate_weights"].shape == (B, 4)

assert torch.isfinite(logits).all()
assert torch.isfinite(
    aux["fused_features"]
).all()

print()
print("✓ REAL FORWARD PASS PASSED")
print("✓ OUTPUT = [B, 40]")
print("✓ NO NaN / INF")

TypeError: BigMultimodalFusion3.forward() got an unexpected keyword argument 'radar'

---
---

In [175]:
# ============================================================
# CELL 63 — CACHED EMBEDDING DATASET
# ============================================================

from torch.utils.data import Dataset, DataLoader


class FusionEmbeddingDataset(Dataset):

    def __init__(self, cache):

        self.skeleton = cache["skeleton"].float()
        self.ir = cache["ir"].float()
        self.depth = cache["depth"].float()
        #self.radar = cache["radar"].float()

        self.labels = cache["labels"].long()

        self.radar_present = (
            cache["radar_present"].bool()
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        return {
            "skeleton": self.skeleton[idx],
            "ir": self.ir[idx],
            "depth": self.depth[idx],
            #"radar": self.radar[idx],

            "radar_present":
                self.radar_present[idx],

            "label":
                self.labels[idx],
        }


train_dataset = FusionEmbeddingDataset(
    train_reload
)

val_dataset = FusionEmbeddingDataset(
    val_reload
)


print("Train:", len(train_dataset))
print("Val:  ", len(val_dataset))


sample = train_dataset[0]

for k, v in sample.items():
    print(
        f"{k:15s}",
        tuple(v.shape) if hasattr(v, "shape") else v
    )

Train: 2238
Val:   693
skeleton        (256,)
ir              (512,)
depth           (128,)
radar_present   ()
label           ()


In [176]:
# ============================================================
# CELL 64 — DATALOADERS
# ============================================================

BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    drop_last=False
)

print("Train batches:", len(train_loader))
print("Val batches:  ", len(val_loader))

Train batches: 35
Val batches:   11


In [177]:
# ============================================================
# CELL 65 — OPTIMIZER / LOSS
# ============================================================

NUM_CLASSES = 40

criterion = nn.CrossEntropyLoss(
    label_smoothing=0.05
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=10,
    eta_min=1e-6
)

print("Optimizer: AdamW")
print("LR:        3e-4")
print("Weight decay:", 1e-4)
print("Label smoothing:", 0.05)

Optimizer: AdamW
LR:        3e-4
Weight decay: 0.0001
Label smoothing: 0.05


In [178]:
# ============================================================
# CELL 66 — METRICS
# ============================================================

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)


def calculate_metrics(y_true, y_pred):

    acc = accuracy_score(
        y_true,
        y_pred
    )

    balanced = balanced_accuracy_score(
        y_true,
        y_pred
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": acc,
        "balanced_accuracy": balanced,
        "macro_f1": macro_f1
    }

In [168]:
# ============================================================
# CELL 67 — WEIGHTS & BIASES
# ============================================================
!pip install wandb -q
import os
import wandb
from kaggle_secrets import UserSecretsClient

# Fetch the secret token safely
user_secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")

# Log into WandB
wandb.login()
import os

try:
    import wandb

    wandb.init(
        project="CIUX",
        name="BIG_FUSION_4MOD_V1 Sk+IR",
        config={
            "architecture": "BigMultimodalFusion",

            "modalities": [
                "Skeleton",
                "IR",
                "Depth",
                "Radar"
            ],

            "hidden_dim": 512,
            "transformer_layers": 3,
            "attention_heads": 8,
            "transformer_ffn": 2048,

            "fusion_params": total_params,

            "batch_size": BATCH_SIZE,
            "epochs": 10,

            "learning_rate": 3e-4,
            "weight_decay": 1e-4,

            "label_smoothing": 0.05,

            "train_samples": 2238,
            "val_samples": 693,

            "radar_train_present": 1110,
            "radar_val_present": 299,
        }
    )

    WANDB_ENABLED = True
    print("✓ W&B enabled")

except Exception as e:

    WANDB_ENABLED = False

    print("W&B unavailable:")
    print(e)
    print("Continuing without W&B.")

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
epoch_time_sec,█▄▆▅▃▆▂▃▂▁▇▅▄▅▂▃▁▄▇▁
gate_depth,▂▃▄▃▃▂▃▄▄▄▄▄▄▄▆▆▇▁█▂
gate_ir,▂▁▆▆▇▄▆▆▆▆▇▆▇█▃▄▅▆▃▄
gate_skeleton,██▃▄▃▅▃▃▃▃▃▃▂▁▅▃▂▅▃▆
lr,▇▇▆▄▃▂▂▁▁▁▂▂▃▄▆▇▇███
train_acc,▁▂▃▅▅▆▇▇███▇▇▇▇▅▆▄▅▅
train_bal_acc,▁▂▄▅▆▆▇▇▇██▇▇▇▆▆▆▅▅▅
train_loss,█▆▅▄▃▂▂▁▁▁▁▁▁▂▂▃▃▄▄▄
train_macro_f1,▁▂▄▅▆▆▇▇███▇▇▇▆▆▆▅▅▅
+4,...


✓ W&B enabled


In [170]:
import warnings
import os
import logging

# 1. Suppress standard Python and library warnings (like deprecation warnings)
warnings.filterwarnings('ignore')

# 2. Suppress warnings from child processes and command-line execution
os.environ["PYTHONWARNINGS"] = "ignore"

# 3. Suppress logging-based warnings from major libraries (like TensorFlow or PyTorch)
logging.getLogger("tensorflow").setLevel(logging.ERROR)
logging.getLogger("pytorch").setLevel(logging.ERROR)

In [179]:
# ============================================================
# CELL — SERIOUS S + IR FUSION TRAINING
# ============================================================

import time
from copy import deepcopy
import torch


# ============================================================
# CONFIG
# ============================================================

EPOCHS = 30

LR = 1e-4
WEIGHT_DECAY = 2e-4

BEST_MODEL_PATH = (
    "/kaggle/working/"
    "serious_sir_fusion_best.pt"
)


# ============================================================
# RESET / CREATE OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)


# ============================================================
# COSINE LR SCHEDULER
# ============================================================

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=1e-6
)


# ============================================================
# LOSS
# ============================================================

criterion = torch.nn.CrossEntropyLoss(
    label_smoothing=0.05
)


# ============================================================
# BEST STATE
# ============================================================

best_val_acc = -1.0
best_epoch = -1
best_state = None

history = []


print("=" * 70)
print("SERIOUS S + IR FUSION TRAINING")
print("=" * 70)

print("Modalities: Skeleton + IR")
print(f"Device:     {device}")
print(f"Epochs:     {EPOCHS}")
print(f"LR:         {LR}")
print(f"Weight decay: {WEIGHT_DECAY}")
print()


# ============================================================
# TRAINING LOOP
# ============================================================

for epoch in range(1, EPOCHS + 1):

    start_time = time.time()


    # ========================================================
    # TRAIN
    # ========================================================

    model.train()

    train_loss = 0.0

    train_true = []
    train_pred = []


    for batch in train_loader:

        # ----------------------------------------------------
        # Inputs
        # ----------------------------------------------------

        skeleton = batch["skeleton"].to(
            device,
            non_blocking=True
        )

        ir = batch["ir"].to(
            device,
            non_blocking=True
        )

        labels = batch["label"].to(
            device,
            non_blocking=True
        )


        # ----------------------------------------------------
        # Clear gradients
        # ----------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )


        # ----------------------------------------------------
        # Forward
        # ----------------------------------------------------

        logits, aux = model(
            skeleton=skeleton,
            ir=ir
        )


        # ----------------------------------------------------
        # Loss
        # ----------------------------------------------------

        loss = criterion(
            logits,
            labels
        )


        # ----------------------------------------------------
        # Backpropagation
        # ----------------------------------------------------

        loss.backward()


        # ----------------------------------------------------
        # Gradient clipping
        # ----------------------------------------------------

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )


        # ----------------------------------------------------
        # Optimizer
        # ----------------------------------------------------

        optimizer.step()


        # ----------------------------------------------------
        # Loss accumulation
        # ----------------------------------------------------

        train_loss += (
            loss.item() *
            labels.size(0)
        )


        # ----------------------------------------------------
        # Predictions
        # ----------------------------------------------------

        predictions = logits.argmax(
            dim=1
        )


        train_true.extend(
            labels.detach()
            .cpu()
            .numpy()
        )

        train_pred.extend(
            predictions.detach()
            .cpu()
            .numpy()
        )


    # ========================================================
    # TRAIN METRICS
    # ========================================================

    train_loss /= len(train_dataset)

    train_metrics = calculate_metrics(
        train_true,
        train_pred
    )


    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    val_loss = 0.0

    val_true = []
    val_pred = []

    gate_values = []


    with torch.no_grad():

        for batch in val_loader:

            # ------------------------------------------------
            # Inputs
            # ------------------------------------------------

            skeleton = batch["skeleton"].to(
                device,
                non_blocking=True
            )

            ir = batch["ir"].to(
                device,
                non_blocking=True
            )

            labels = batch["label"].to(
                device,
                non_blocking=True
            )


            # ------------------------------------------------
            # Forward
            # ------------------------------------------------

            logits, aux = model(
                skeleton=skeleton,
                ir=ir
            )


            # ------------------------------------------------
            # Loss
            # ------------------------------------------------

            loss = criterion(
                logits,
                labels
            )


            val_loss += (
                loss.item() *
                labels.size(0)
            )


            # ------------------------------------------------
            # Predictions
            # ------------------------------------------------

            predictions = logits.argmax(
                dim=1
            )


            val_true.extend(
                labels.cpu()
                .numpy()
            )

            val_pred.extend(
                predictions.cpu()
                .numpy()
            )


            # ------------------------------------------------
            # Gate values
            #
            # [Skeleton, IR]
            # ------------------------------------------------

            gate_values.append(
                aux["gate_weights"]
                .cpu()
            )


    # ========================================================
    # VALIDATION METRICS
    # ========================================================

    val_loss /= len(val_dataset)

    val_metrics = calculate_metrics(
        val_true,
        val_pred
    )


    # ========================================================
    # LR STEP
    # ========================================================

    scheduler.step()


    # ========================================================
    # GATE ANALYSIS
    # ========================================================

    gate_values = torch.cat(
        gate_values,
        dim=0
    )

    mean_gates = (
        gate_values
        .mean(dim=0)
        .numpy()
    )


    # ========================================================
    # BEST MODEL
    # ========================================================

    if (
        val_metrics["accuracy"]
        >
        best_val_acc
    ):

        best_val_acc = (
            val_metrics["accuracy"]
        )

        best_epoch = epoch

        best_state = deepcopy(
            model.state_dict()
        )


    # ========================================================
    # TIME
    # ========================================================

    elapsed = (
        time.time() -
        start_time
    )


    # ========================================================
    # HISTORY
    # ========================================================

    row = {

        "epoch": epoch,

        # ----------------------------------------------------
        # Train
        # ----------------------------------------------------

        "train_loss":
            train_loss,

        "train_acc":
            train_metrics["accuracy"],

        "train_bal_acc":
            train_metrics["balanced_accuracy"],

        "train_macro_f1":
            train_metrics["macro_f1"],

        # ----------------------------------------------------
        # Validation
        # ----------------------------------------------------

        "val_loss":
            val_loss,

        "val_acc":
            val_metrics["accuracy"],

        "val_bal_acc":
            val_metrics["balanced_accuracy"],

        "val_macro_f1":
            val_metrics["macro_f1"],

        # ----------------------------------------------------
        # Gates
        # ----------------------------------------------------

        "gate_skeleton":
            float(mean_gates[0]),

        "gate_ir":
            float(mean_gates[1]),

        # ----------------------------------------------------
        # Optimization
        # ----------------------------------------------------

        "lr":
            optimizer.param_groups[0]["lr"],

        "epoch_time_sec":
            elapsed
    }


    history.append(row)


    # ========================================================
    # W&B
    # ========================================================

    if WANDB_ENABLED:

        wandb.log(
            row
        )


    # ========================================================
    # PRINT
    # ========================================================

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train "
        f"{train_metrics['accuracy'] * 100:.2f}% | "
        f"Val "
        f"{val_metrics['accuracy'] * 100:.2f}% | "
        f"Bal "
        f"{val_metrics['balanced_accuracy'] * 100:.2f}% | "
        f"F1 "
        f"{val_metrics['macro_f1'] * 100:.2f}% | "
        f"Gates "
        f"[{mean_gates[0]:.2f}, "
        f"{mean_gates[1]:.2f}] | "
        f"LR "
        f"{optimizer.param_groups[0]['lr']:.2e} | "
        f"{elapsed:.1f}s"
    )


# ============================================================
# RESTORE BEST
# ============================================================

if best_state is not None:

    model.load_state_dict(
        best_state
    )


# ============================================================
# SAVE BEST CHECKPOINT
# ============================================================

torch.save(
    {
        "model_state_dict":
            model.state_dict(),

        "best_val_acc":
            best_val_acc,

        "best_epoch":
            best_epoch,

        "history":
            history,

        "modalities":
            [
                "skeleton",
                "ir"
            ],

        "skeleton_dim":
            256,

        "ir_dim":
            512,

        "num_classes":
            40,

        "model_parameters":
            sum(
                p.numel()
                for p in model.parameters()
            )
    },
    BEST_MODEL_PATH
)


# ============================================================
# FINAL
# ============================================================

print()
print("=" * 70)
print("✓ SERIOUS S + IR TRAINING COMPLETE")
print("=" * 70)

print(
    f"Best validation accuracy: "
    f"{best_val_acc * 100:.2f}%"
)

print(
    f"Best epoch: "
    f"{best_epoch}"
)

print(
    f"Parameters: "
    f"{sum(p.numel() for p in model.parameters()):,}"
)

print(
    f"FP32 size: "
    f"{sum(p.numel() for p in model.parameters()) * 4 / 1024**2:.2f} MB"
)

print()
print("Saved:")
print(BEST_MODEL_PATH)

print("=" * 70)

SERIOUS S + IR FUSION TRAINING
Modalities: Skeleton + IR
Device:     cuda
Epochs:     30
LR:         0.0001
Weight decay: 0.0002

Epoch 01/30 | Train 39.54% | Val 59.60% | Bal 45.24% | F1 44.96% | Gates [0.52, 0.48] | LR 9.97e-05 | 1.1s
Epoch 02/30 | Train 73.95% | Val 64.79% | Bal 56.26% | F1 55.92% | Gates [0.52, 0.48] | LR 9.89e-05 | 1.1s
Epoch 03/30 | Train 83.29% | Val 67.10% | Bal 60.76% | F1 60.22% | Gates [0.52, 0.48] | LR 9.76e-05 | 1.1s
Epoch 04/30 | Train 88.07% | Val 67.53% | Bal 61.35% | F1 60.23% | Gates [0.51, 0.49] | LR 9.57e-05 | 1.1s
Epoch 05/30 | Train 89.54% | Val 65.95% | Bal 61.44% | F1 60.19% | Gates [0.51, 0.49] | LR 9.34e-05 | 1.2s
Epoch 06/30 | Train 91.51% | Val 67.10% | Bal 62.84% | F1 62.59% | Gates [0.51, 0.49] | LR 9.05e-05 | 1.1s
Epoch 07/30 | Train 93.03% | Val 65.22% | Bal 60.99% | F1 60.23% | Gates [0.51, 0.49] | LR 8.73e-05 | 1.1s
Epoch 08/30 | Train 93.16% | Val 63.64% | Bal 59.91% | F1 59.12% | Gates [0.52, 0.48] | LR 8.36e-05 | 1.1s
Epoch 09/30 | 

In [186]:
checkpoint = torch.load(
    "/kaggle/working/serious_sir_fusion_best.pt",
    map_location=device,
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.to(device)
model.eval()

print("Loaded best epoch:", checkpoint["best_epoch"])
print(
    "Best validation accuracy:",
    checkpoint["best_val_acc"] * 100,
    "%"
)

Loaded best epoch: 4
Best validation accuracy: 67.53246753246754 %


In [187]:
# ============================================================
# S + IR TEST EMBEDDINGS
# ============================================================

model.eval()

test_skeleton_emb = test_skeleton_emb.to(device)
test_ir_emb = test_ir_emb.to(device)

print("Skeleton:", test_skeleton_emb.shape)
print("IR:", test_ir_emb.shape)

with torch.no_grad():

    logits, aux = model(
        skeleton=test_skeleton_emb,
        ir=test_ir_emb
    )

    predictions = logits.argmax(
        dim=1
    )

print("Logits:", logits.shape)
print("Predictions:", predictions.shape)

print()
print("Prediction classes:")
print(
    torch.unique(
        predictions,
        return_counts=True
    )
)

Skeleton: torch.Size([405, 256])
IR: torch.Size([405, 512])
Logits: torch.Size([405, 40])
Predictions: torch.Size([405])

Prediction classes:
(tensor([ 0,  1,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 17, 18, 20, 21, 22, 23,
        24, 26, 27, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39],
       device='cuda:0'), tensor([11,  2,  4, 10,  9, 25, 40, 72, 29,  8,  6,  3, 13,  1, 14,  1,  3,  1,
         3, 47,  7,  9,  4,  8,  3,  1,  9, 10, 42,  4,  2,  4],
       device='cuda:0'))


In [188]:
# ============================================================
# CREATE SUBMISSION
# ============================================================

all_predictions = (
    predictions
    .cpu()
    .numpy()
    .tolist()
)

all_trial_ids = test_ids

submission = pd.DataFrame({
    "path": [
        f"small_model_track_test/{trial_id}/"
        for trial_id in all_trial_ids
    ],
    "prediction": all_predictions
})

print(submission.head(10))
print()
print("Shape:", submission.shape)
print("Columns:", submission.columns.tolist())

submission.to_csv(
    "submission28.csv",
    index=False
)

print()
print("✓ submission.csv created")

                                   path  prediction
0  small_model_track_test/SM_test_0001/          36
1  small_model_track_test/SM_test_0002/          35
2  small_model_track_test/SM_test_0003/          10
3  small_model_track_test/SM_test_0004/           8
4  small_model_track_test/SM_test_0005/          10
5  small_model_track_test/SM_test_0006/           9
6  small_model_track_test/SM_test_0007/          26
7  small_model_track_test/SM_test_0008/          30
8  small_model_track_test/SM_test_0009/           7
9  small_model_track_test/SM_test_0010/           9

Shape: (405, 2)
Columns: ['path', 'prediction']

✓ submission.csv created


---
---

In [129]:
# ============================================================
# FINAL TEST INFERENCE — 405 REAL TEST TRIALS
# ============================================================

import json
import cv2
import numpy as np
import pandas as pd
import torch
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm


model.eval()
skeleton_encoder.eval()
ir_encoder.eval()
depth_encoder.eval()
radar_encoder.eval()


# ============================================================
# RADAR — EXACT TEST LOADER
# ============================================================

RADAR_FEATURE_COLS = [
    "x", "y", "z", "v", "snr", "noise"
]


def load_test_radar_exact(csv_path):

    x = np.zeros(
        (
            128,
            32,
            6
        ),
        dtype=np.float32
    )

    frame_mask = np.zeros(
        128,
        dtype=bool
    )

    if csv_path is None:
        return x, frame_mask

    try:
        df = pd.read_csv(csv_path)
    except Exception:
        return x, frame_mask

    if len(df) == 0:
        return x, frame_mask

    # Exact six training features
    values = df[
        RADAR_FEATURE_COLS
    ].apply(
        pd.to_numeric,
        errors="coerce"
    )

    frame_ids = pd.to_numeric(
        df["frame"],
        errors="coerce"
    )

    valid = (
        values.notna().all(axis=1)
        &
        frame_ids.notna()
    )

    values = values.loc[valid].to_numpy(
        dtype=np.float32
    )

    frame_ids = frame_ids.loc[valid].to_numpy()

    if len(values) == 0:
        return x, frame_mask

    unique_frames = np.unique(
        frame_ids
    )

    # Same maximum-frame logic
    if len(unique_frames) > 128:

        start = (
            len(unique_frames) - 128
        ) // 2

        unique_frames = unique_frames[
            start:start + 128
        ]

    # Populate
    for out_idx, frame_id in enumerate(
        unique_frames
    ):

        detections = values[
            frame_ids == frame_id
        ]

        if len(detections) == 0:
            continue

        # Highest SNR, exactly as training
        if len(detections) > 32:

            order = np.argsort(
                detections[:, 4]
            )[::-1]

            detections = detections[
                order[:32]
            ]

        n = min(
            len(detections),
            32
        )

        x[
            out_idx,
            :n
        ] = detections[:n]

        frame_mask[
            out_idx
        ] = True

    # Exact training normalization
    mean = np.array([
        -0.173170313,
         1.08870163,
        -0.168657074,
        -0.000711193,
       173.842025,
       526.117324
    ], dtype=np.float32)

    std = np.array([
        0.88422244,
        1.18826219,
        1.24050746,
        0.15830402,
       45.52745306,
       65.53307942
    ], dtype=np.float32)

    # Only normalize populated detections.
    populated = np.any(
        x != 0,
        axis=-1
    )

    x[populated] = (
        x[populated] - mean
    ) / (
        std + 1e-6
    )

    return x, frame_mask


# ============================================================
# IR
# ============================================================

def load_test_ir_exact(trial_dir):

    files = sorted(
        (trial_dir / "IR").glob("*.png"),
        key=lambda p: int(
            p.stem.split("_")[-1]
        )
    )

    if len(files) == 0:
        return (
            torch.zeros(
                24, 3, 120, 160
            ),
            torch.zeros(24).bool()
        )

    n = len(files)

    indices = np.linspace(
        0,
        n - 1,
        24
    ).round().astype(int)

    frames = []

    valid_mask = []

    for idx in indices:

        try:

            img = cv2.imread(
                str(files[idx]),
                cv2.IMREAD_GRAYSCALE
            )

            if img is None:
                raise ValueError()

            img = img.astype(
                np.float32
            )

            lo, hi = np.percentile(
                img,
                [2, 98]
            )

            if hi > lo:

                raw = np.clip(
                    (img - lo) /
                    (hi - lo),
                    0,
                    1
                )

            else:

                raw = img / 255.0

            raw = cv2.resize(
                raw,
                (160, 120),
                interpolation=cv2.INTER_AREA
            )

            gx = cv2.Sobel(
                raw,
                cv2.CV_32F,
                1,
                0,
                ksize=3
            )

            gy = cv2.Sobel(
                raw,
                cv2.CV_32F,
                0,
                1,
                ksize=3
            )

            grad = np.sqrt(
                gx * gx + gy * gy
            )

            grad /= (
                grad.max() + 1e-6
            )

            frames.append(
                raw
            )

            valid_mask.append(True)

        except Exception:

            frames.append(
                np.zeros(
                    (120,160),
                    dtype=np.float32
                )
            )

            valid_mask.append(False)

    raw_stack = np.stack(
        frames
    )

    differences = np.zeros_like(
        raw_stack
    )

    differences[1:] = (
        raw_stack[1:]
        -
        raw_stack[:-1]
    )

    # Recompute gradients
    gradients = []

    for raw in raw_stack:

        gx = cv2.Sobel(
            raw,
            cv2.CV_32F,
            1,
            0,
            ksize=3
        )

        gy = cv2.Sobel(
            raw,
            cv2.CV_32F,
            0,
            1,
            ksize=3
        )

        g = np.sqrt(
            gx * gx + gy * gy
        )

        g /= g.max() + 1e-6

        gradients.append(g)

    gradients = np.stack(
        gradients
    )

    out = np.stack(
        [
            raw_stack,
            gradients,
            differences
        ],
        axis=1
    )

    return (
        torch.from_numpy(
            out.astype(np.float32)
        ),
        torch.tensor(
            valid_mask
        )
    )


# ============================================================
# DEPTH / COLOR
# ============================================================

def load_test_depth_exact(trial_dir):

    files = sorted(
        (trial_dir / "Depth_Color").glob("*.png"),
        key=lambda p: int(
            p.stem.split("_")[-2]
        )
    )

    frames = []
    valid = []

    if len(files) > 32:

        indices = np.linspace(
            0,
            len(files) - 1,
            32
        ).round().astype(int)

        files = [
            files[i]
            for i in indices
        ]

    for p in files:

        try:

            img = Image.open(
                p
            ).convert("RGB")

            img = img.resize(
                (160,160),
                Image.Resampling.BILINEAR
            )

            arr = np.asarray(
                img,
                dtype=np.float32
            ) / 255.0

            # ImageNet normalization
            mean = np.array(
                [0.485, 0.456, 0.406],
                dtype=np.float32
            )

            std = np.array(
                [0.229, 0.224, 0.225],
                dtype=np.float32
            )

            arr = (
                arr - mean
            ) / std

            arr = np.transpose(
                arr,
                (2,0,1)
            )

            frames.append(arr)
            valid.append(True)

        except Exception:

            frames.append(
                np.zeros(
                    (3,160,160),
                    dtype=np.float32
                )
            )

            valid.append(False)

    while len(frames) < 32:

        frames.append(
            np.zeros(
                (3,160,160),
                dtype=np.float32
            )
        )

        valid.append(False)

    frames = frames[:32]
    valid = valid[:32]

    return (
        torch.from_numpy(
            np.stack(frames).astype(
                np.float32
            )
        ),
        torch.tensor(valid)
    )


# ============================================================
# PROCESS ONE TRIAL
# ============================================================

def encode_test_trial(trial_dir):

    # --------------------------------------------------------
    # Skeleton
    # --------------------------------------------------------

    skel = load_test_skeleton(
        trial_dir / "Skeleton"
    )

    skel = torch.from_numpy(
        skel
    ).unsqueeze(0).to(device)

    with torch.no_grad():

        s_emb = skeleton_encoder(
            skel
        )


    # --------------------------------------------------------
    # IR
    # --------------------------------------------------------

    ir, ir_mask = load_test_ir_exact(
        trial_dir
    )

    ir = ir.unsqueeze(0).to(device)
    ir_mask = ir_mask.unsqueeze(0).to(device)

    with torch.no_grad():

        i_emb = ir_encoder(
            ir,
            mask=ir_mask
        )


    # --------------------------------------------------------
    # Depth
    # --------------------------------------------------------

    depth, depth_mask = load_test_depth_exact(
        trial_dir
    )

    depth = depth.unsqueeze(0).to(device)
    depth_mask = depth_mask.unsqueeze(0).to(device)

    with torch.no_grad():

        d_emb = depth_encoder(
            depth,
            mask=depth_mask
        )


    # --------------------------------------------------------
    # Radar
    # --------------------------------------------------------

    radar_files = sorted(
        (trial_dir / "Radar").glob("*.csv")
    )

    radar_path = (
        radar_files[0]
        if radar_files
        else None
    )

    radar_np, radar_mask_np = (
        load_test_radar_exact(
            radar_path
        )
    )

    radar = torch.from_numpy(
        radar_np
    ).unsqueeze(0).to(device)

    radar_mask = torch.from_numpy(
        radar_mask_np
    ).unsqueeze(0).to(device)

    radar_present = (
        radar_mask.any(
            dim=1
        )
    )

    with torch.no_grad():

        r_emb = radar_encoder(
            radar,
            frame_mask=radar_mask
        )


    # --------------------------------------------------------
    # BIG FUSION
    # --------------------------------------------------------

    with torch.no_grad():

        logits, aux = model(
            skeleton=s_emb,
            ir=i_emb,
            depth=d_emb,
            radar=r_emb,
            radar_present=radar_present
        )

        prediction = (
            logits.argmax(
                dim=1
            ).item()
        )
    return (
    prediction,
    bool(radar_present.item()),
    s_emb.detach().cpu().squeeze(0),
    i_emb.detach().cpu().squeeze(0),
    d_emb.detach().cpu().squeeze(0),
    r_emb.detach().cpu().squeeze(0),
)


# ============================================================
# TEST FIRST TRIAL
# ============================================================

result = encode_test_trial(
    TEST_ROOT / "SM_test_0001"
)

print("=" * 70)
print("SM_test_0001 COMPLETE PIPELINE")
print("=" * 70)

print("Prediction:", result[0])
print("Radar present:", result[1])
print("Skeleton:", result[2])
print("IR:", result[3])
print("Depth:", result[4])
print("Radar:", result[5])

print()
print("✓ RAW → 4 ENCODERS → BIG FUSION WORKS")

SM_test_0001 COMPLETE PIPELINE
Prediction: 36
Radar present: False
Skeleton: tensor([ 0.0497, -2.4206,  1.6072, -1.4613,  0.3394, -0.5916,  2.0589, -1.4085,
         2.5953, -0.0064, -0.3250,  0.3315,  0.8361, -2.2594,  0.2864,  0.9359,
        -1.3375, -1.0963, -1.8245, -0.0976, -0.8671, -1.2844, -0.1140, -0.0250,
        -1.5244,  0.5595, -0.7286, -1.6336,  0.0878,  0.6896, -0.1497, -0.6346,
        -0.3784, -0.8612, -0.4278, -0.5349,  0.6346,  0.3095, -0.9946, -1.5648,
         0.9491, -1.0406, -0.4277, -0.3155, -0.6774,  1.3535,  1.3715, -0.0565,
         0.7933, -0.3190, -0.4177,  0.0783, -2.6107, -0.2587, -0.1937, -1.4879,
         1.3134,  1.2653,  1.0885, -0.0966,  1.0875, -0.3277,  0.5705, -0.7133,
         0.5712, -0.0986, -1.5849, -0.3354, -0.1184, -0.6028,  1.9692,  2.3306,
         1.0488, -0.0157, -1.9869,  0.3569,  0.8118,  2.0825, -0.4898, -1.7945,
         0.0152,  0.1125,  3.3197,  0.5867,  1.2366,  0.8698, -0.1104, -0.2318,
        -0.9675,  1.0280,  0.8838,  1.0274,

In [130]:
# ============================================================
# COLLECT TEST EMBEDDINGS
# ============================================================

test_skeleton_emb = []
test_ir_emb = []
test_depth_emb = []
test_radar_emb = []

test_predictions = []
test_radar_present = []
test_ids = []

for trial_dir in tqdm(
    test_dataset.trial_dirs,
    desc="Extracting test embeddings"
):

    (
        pred,
        radar_ok,
        s,
        i,
        d,
        r
    ) = encode_test_trial(trial_dir)

    test_ids.append(trial_dir.name)

    test_predictions.append(pred)
    test_radar_present.append(radar_ok)

    test_skeleton_emb.append(s)
    test_ir_emb.append(i)
    test_depth_emb.append(d)
    test_radar_emb.append(r)


test_skeleton_emb = torch.stack(
    test_skeleton_emb
)

test_ir_emb = torch.stack(
    test_ir_emb
)

test_depth_emb = torch.stack(
    test_depth_emb
)

test_radar_emb = torch.stack(
    test_radar_emb
)

test_radar_present = torch.tensor(
    test_radar_present
)

print("Skeleton:", test_skeleton_emb.shape)
print("IR:      ", test_ir_emb.shape)
print("Depth:   ", test_depth_emb.shape)
print("Radar:   ", test_radar_emb.shape)

Extracting test embeddings:   0%|          | 0/405 [00:00<?, ?it/s]

Skeleton: torch.Size([405, 256])
IR:       torch.Size([405, 512])
Depth:    torch.Size([405, 128])
Radar:    torch.Size([405, 256])


In [131]:
# ============================================================
# TRAIN vs TEST EMBEDDING HEALTH
# ============================================================

def stats(name, train_x, test_x):

    print()
    print("=" * 65)
    print(name)
    print("=" * 65)

    print(
        f"{'':15s}"
        f"{'TRAIN':>15s}"
        f"{'TEST':>15s}"
    )

    for label, fn in [
        ("mean", lambda x: x.mean().item()),
        ("std",  lambda x: x.std().item()),
        ("min",  lambda x: x.min().item()),
        ("max",  lambda x: x.max().item()),
        ("abs mean", lambda x: x.abs().mean().item()),
    ]:

        print(
            f"{label:15s}"
            f"{fn(train_x):15.5f}"
            f"{fn(test_x):15.5f}"
        )


stats(
    "SKELETON",
    train_reload["skeleton"],
    test_skeleton_emb
)

stats(
    "IR",
    train_reload["ir"],
    test_ir_emb
)

stats(
    "DEPTH",
    train_reload["depth"],
    test_depth_emb
)

stats(
    "RADAR",
    train_reload["radar"][
        train_reload["radar_present"]
    ],
    test_radar_emb[
        test_radar_present
    ]
)


SKELETON
                         TRAIN           TEST
mean                  -0.00003        0.00001
std                    1.04456        1.03988
min                   -4.95387       -4.30982
max                    4.40034        4.54076
abs mean               0.83117        0.82701

IR
                         TRAIN           TEST
mean                  -0.00051        0.02085
std                    0.24716        0.22252
min                   -0.95133       -0.75867
max                    0.94683        0.74708
abs mean               0.17398        0.16109

DEPTH
                         TRAIN           TEST
mean                   1.22928        1.06174
std                    2.08490        1.83867
min                   -0.27822       -0.27825
max                   19.26890       15.41853
abs mean               1.35966        1.18932

RADAR
                         TRAIN           TEST
mean                   2.08930            nan
std                    2.47850            nan
min   

In [132]:
# ============================================================
# PREDICTION DISTRIBUTION
# ============================================================

from collections import Counter

counts = Counter(
    test_predictions
)

print(
    "Unique predicted classes:",
    len(counts)
)

print()

for cls, n in sorted(
    counts.items()
):

    print(
        f"{cls:2d}: {n:3d} "
        f"({n / 405 * 100:5.1f}%)"
    )

Unique predicted classes: 28

 0: 202 ( 49.9%)
 2:   1 (  0.2%)
 4:   7 (  1.7%)
 6:   2 (  0.5%)
 7:  15 (  3.7%)
 8:   6 (  1.5%)
10:   7 (  1.7%)
11:   2 (  0.5%)
12:  15 (  3.7%)
13:   1 (  0.2%)
14:   4 (  1.0%)
17:   2 (  0.5%)
18:   3 (  0.7%)
19:   1 (  0.2%)
20:   3 (  0.7%)
21:   1 (  0.2%)
22:  11 (  2.7%)
24:   1 (  0.2%)
26:  55 ( 13.6%)
27:  19 (  4.7%)
29:   2 (  0.5%)
31:  12 (  3.0%)
32:   2 (  0.5%)
34:  12 (  3.0%)
35:   1 (  0.2%)
36:  15 (  3.7%)
37:   2 (  0.5%)
39:   1 (  0.2%)


In [133]:
# ============================================================
# TEST BIG FUSION WITH RADAR FORCED OFF
# ============================================================

model.eval()

pred_no_radar = []

with torch.no_grad():

    for start in range(
        0,
        405,
        64
    ):

        end = min(
            start + 64,
            405
        )

        logits, _ = model(
            skeleton=test_skeleton_emb[start:end].to(device),
            ir=test_ir_emb[start:end].to(device),
            depth=test_depth_emb[start:end].to(device),

            radar=torch.zeros_like(
                test_radar_emb[start:end]
            ).to(device),

            radar_present=torch.zeros(
                end - start,
                dtype=torch.bool,
                device=device
            )
        )

        pred_no_radar.extend(
            logits.argmax(
                dim=1
            ).cpu().tolist()
        )


print(
    "Normal predictions:",
    test_predictions[:10]
)

print(
    "No-Radar predictions:",
    pred_no_radar[:10]
)

print(
    "Changed predictions:",
    sum(
        a != b
        for a, b in zip(
            test_predictions,
            pred_no_radar
        )
    ),
    "/ 405"
)

Normal predictions: [36, 0, 7, 29, 10, 0, 26, 0, 0, 0]
No-Radar predictions: [36, 26, 7, 29, 10, 18, 26, 30, 7, 26]
Changed predictions: 193 / 405


In [140]:
# ============================================================
# CORRECT RADAR TEST LOADER
# ============================================================

RADAR_FEATURES = ["x", "y", "z", "v", "snr", "noise"]

def load_test_radar_fixed(trial_id):
    trial_dir = TEST_ROOT / trial_id
    radar_dir = trial_dir / "Radar"

    csv_files = list(radar_dir.glob("*.csv"))

    # Default empty sequence
    sequence = np.zeros((128, 32, 6), dtype=np.float32)
    frame_mask = np.zeros(128, dtype=bool)

    if not csv_files:
        return sequence, frame_mask

    # Find the actual Radar CSV
    radar_csv = None

    for p in csv_files:
        try:
            tmp = pd.read_csv(p)
            if all(c in tmp.columns for c in RADAR_FEATURES) and "frame" in tmp.columns:
                radar_csv = p
                break
        except Exception:
            continue

    if radar_csv is None:
        return sequence, frame_mask

    df_r = pd.read_csv(radar_csv)

    # Empty CSV
    if len(df_r) == 0:
        return sequence, frame_mask

    # Numeric conversion — EXACT six features
    for c in RADAR_FEATURES:
        df_r[c] = pd.to_numeric(df_r[c], errors="coerce")

    df_r["frame"] = pd.to_numeric(df_r["frame"], errors="coerce")

    df_r = df_r.dropna(
        subset=["frame"] + RADAR_FEATURES
    ).copy()

    if len(df_r) == 0:
        return sequence, frame_mask

    # Sort by frame
    df_r = df_r.sort_values("frame")

    frame_ids = df_r["frame"].unique()

    # --------------------------------------------------------
    # Temporal selection: max 128 frames
    # --------------------------------------------------------
    if len(frame_ids) > 128:
        # Center window for deterministic test inference
        start = (len(frame_ids) - 128) // 2
        frame_ids = frame_ids[start:start + 128]

    # --------------------------------------------------------
    # Fill each frame
    # --------------------------------------------------------
    for t, frame_id in enumerate(frame_ids):

        if t >= 128:
            break

        frame_df = df_r[df_r["frame"] == frame_id]

        # Highest-SNR 32 detections
        if len(frame_df) > 32:
            frame_df = frame_df.nlargest(32, "snr")

        values = frame_df[RADAR_FEATURES].to_numpy(
            dtype=np.float32
        )

        n = min(len(values), 32)

        if n > 0:
            sequence[t, :n] = values[:n]
            frame_mask[t] = True

    # --------------------------------------------------------
    # Normalize EXACTLY like training
    # --------------------------------------------------------
    sequence = (
        sequence - radar_mean_np.reshape(1, 1, 6)
    ) / radar_std_np.reshape(1, 1, 6)

    # Keep padding zero AFTER normalization
    sequence[~frame_mask] = 0.0

    return sequence.astype(np.float32), frame_mask

x_np, mask_np = load_test_radar_fixed("SM_test_0002")

print("Shape:", x_np.shape)
print("Valid frames:", mask_np.sum())
print("NaN:", np.isnan(x_np).sum())
print("Inf:", np.isinf(x_np).sum())
print("Non-zero detections:", np.count_nonzero(np.abs(x_np).sum(axis=-1)))

print("\nFirst valid frame:")
first = np.where(mask_np)[0][0]
print(x_np[first, :5])

NameError: name 'radar_mean_np' is not defined

In [135]:
# ============================================================
# NO-RADAR TEST PREDICTIONS
# ============================================================

model.eval()

pred_no_radar = []

with torch.no_grad():

    for start in range(
        0,
        405,
        64
    ):

        end = min(
            start + 64,
            405
        )

        logits, _ = model(
            skeleton=
                test_skeleton_emb[start:end].to(device),

            ir=
                test_ir_emb[start:end].to(device),

            depth=
                test_depth_emb[start:end].to(device),

            radar=
                torch.zeros_like(
                    test_radar_emb[start:end]
                ).to(device),

            radar_present=
                torch.zeros(
                    end - start,
                    dtype=torch.bool,
                    device=device
                )
        )

        pred_no_radar.extend(
            logits.argmax(
                dim=1
            ).cpu().tolist()
        )


print(
    "No-Radar predictions:",
    len(pred_no_radar)
)

print(
    pred_no_radar[:20]
)

No-Radar predictions: 405
[36, 26, 7, 29, 10, 18, 26, 30, 7, 26, 20, 4, 29, 4, 13, 17, 38, 27, 31, 26]


In [136]:
# ============================================================
# NO-RADAR SUBMISSION
# ============================================================

submission_no_radar = pd.DataFrame({
    "path": [
        f"small_model_track_test/{trial_id}/"
        for trial_id in test_ids
    ],
    "prediction": pred_no_radar
})

NO_RADAR_PATH = (
    "/kaggle/working/"
    "submission_no_radar.csv"
)

submission_no_radar.to_csv(
    NO_RADAR_PATH,
    index=False
)

print(
    submission_no_radar.head()
)

print(
    submission_no_radar.shape
)

print(
    f"Saved: {NO_RADAR_PATH}"
)

                                   path  prediction
0  small_model_track_test/SM_test_0001/          36
1  small_model_track_test/SM_test_0002/          26
2  small_model_track_test/SM_test_0003/           7
3  small_model_track_test/SM_test_0004/          29
4  small_model_track_test/SM_test_0005/          10
(405, 2)
Saved: /kaggle/working/submission_no_radar.csv


In [ ]:
# ============================================================
# TRAIN vs TEST EMBEDDING HEALTH
# ============================================================

def stats(name, train_x, test_x):

    print()
    print("=" * 65)
    print(name)
    print("=" * 65)

    print(
        f"{'':15s}"
        f"{'TRAIN':>15s}"
        f"{'TEST':>15s}"
    )

    for label, fn in [
        ("mean", lambda x: x.mean().item()),
        ("std",  lambda x: x.std().item()),
        ("min",  lambda x: x.min().item()),
        ("max",  lambda x: x.max().item()),
        ("abs mean", lambda x: x.abs().mean().item()),
    ]:

        print(
            f"{label:15s}"
            f"{fn(train_x):15.5f}"
            f"{fn(test_x):15.5f}"
        )


stats(
    "SKELETON",
    train_reload["skeleton"],
    test_skeleton_emb
)

stats(
    "IR",
    train_reload["ir"],
    test_ir_emb
)

stats(
    "DEPTH",
    train_reload["depth"],
    test_depth_emb
)

stats(
    "RADAR",
    train_reload["radar"][
        train_reload["radar_present"]
    ],
    test_radar_emb[
        test_radar_present
    ]
)

In [127]:
# ============================================================
# FINAL — INFERENCE ON ALL 405 TEST TRIALS
# ============================================================

all_predictions = []
all_trial_ids = []
radar_status = []

print("=" * 70)
print("RUNNING FINAL TEST INFERENCE")
print("=" * 70)

for trial_dir in tqdm(
    test_dataset.trial_dirs,
    total=len(test_dataset.trial_dirs),
    desc="Test trials"
):

    prediction, radar_present, *_ = (
        encode_test_trial(
            trial_dir
        )
    )

    all_trial_ids.append(
        trial_dir.name
    )

    all_predictions.append(
        prediction
    )

    radar_status.append(
        radar_present
    )


print()
print("=" * 70)
print("INFERENCE COMPLETE")
print("=" * 70)

print(
    "Predictions:",
    len(all_predictions)
)

print(
    "Trial IDs:",
    len(all_trial_ids)
)

print(
    "Radar available:",
    sum(radar_status),
    "/",
    len(radar_status)
)

assert len(all_predictions) == 405
assert len(all_trial_ids) == 405

print()
print("First 10:")

for tid, pred in zip(
    all_trial_ids[:10],
    all_predictions[:10]
):

    print(
        tid,
        "->",
        pred
    )

RUNNING FINAL TEST INFERENCE


Test trials:   0%|          | 0/405 [00:00<?, ?it/s]


INFERENCE COMPLETE
Predictions: 405
Trial IDs: 405
Radar available: 198 / 405

First 10:
SM_test_0001 -> 36
SM_test_0002 -> 0
SM_test_0003 -> 7
SM_test_0004 -> 29
SM_test_0005 -> 10
SM_test_0006 -> 0
SM_test_0007 -> 26
SM_test_0008 -> 0
SM_test_0009 -> 0
SM_test_0010 -> 0


In [128]:
# ============================================================
# CREATE FINAL SUBMISSION
# ============================================================

from collections import Counter

prediction_counts = Counter(
    all_predictions
)

print("=" * 70)
print("PREDICTION DISTRIBUTION")
print("=" * 70)

for label, count in sorted(
    prediction_counts.items()
):

    print(
        f"{label:2d}: {count}"
    )


# ------------------------------------------------------------
# EXACT CUHK-X SUBMISSION FORMAT
# ------------------------------------------------------------

submission = pd.DataFrame({
    "path": [
        f"small_model_track_test/{trial_id}/"
        for trial_id in all_trial_ids
    ],
    "prediction": all_predictions
})


print()
print("=" * 70)
print("SUBMISSION")
print("=" * 70)

print(
    submission.head(10)
)

print()
print(
    "Shape:",
    submission.shape
)

print(
    "Columns:",
    submission.columns.tolist()
)


assert submission.shape == (
    405,
    2
)

assert submission.columns.tolist() == [
    "path",
    "prediction"
]

assert submission["path"].is_unique

assert all(
    0 <= int(x) < 40
    for x in submission["prediction"]
)


# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

SUBMISSION_PATH = (
    "/kaggle/working/submission7.csv"
)

submission.to_csv(
    SUBMISSION_PATH,
    index=False
)

print()
print(
    f"✓ SAVED: {SUBMISSION_PATH}"
)

print(
    f"Size: "
    f"{Path(SUBMISSION_PATH).stat().st_size / 1024:.2f} KB"
)

PREDICTION DISTRIBUTION
 0: 202
 2: 1
 4: 7
 6: 2
 7: 15
 8: 6
10: 7
11: 2
12: 15
13: 1
14: 4
17: 2
18: 3
19: 1
20: 3
21: 1
22: 11
24: 1
26: 55
27: 19
29: 2
31: 12
32: 2
34: 12
35: 1
36: 15
37: 2
39: 1

SUBMISSION
                                   path  prediction
0  small_model_track_test/SM_test_0001/          36
1  small_model_track_test/SM_test_0002/           0
2  small_model_track_test/SM_test_0003/           7
3  small_model_track_test/SM_test_0004/          29
4  small_model_track_test/SM_test_0005/          10
5  small_model_track_test/SM_test_0006/           0
6  small_model_track_test/SM_test_0007/          26
7  small_model_track_test/SM_test_0008/           0
8  small_model_track_test/SM_test_0009/           0
9  small_model_track_test/SM_test_0010/           0

Shape: (405, 2)
Columns: ['path', 'prediction']

✓ SAVED: /kaggle/working/submission7.csv
Size: 15.61 KB


In [123]:
# ============================================================
# FINAL SUBMISSION VERIFICATION
# ============================================================

check = pd.read_csv(
    "/kaggle/working/submission7.csv"
)

print(check.head())
print()
print("Rows:", len(check))
print("Columns:", check.columns.tolist())
print("Unique paths:", check["path"].nunique())
print(
    "Prediction range:",
    check["prediction"].min(),
    "to",
    check["prediction"].max()
)

assert len(check) == 405
assert check["path"].nunique() == 405
assert check["prediction"].between(0, 39).all()

print()
print("============================================================")
print("✓ SUBMISSION CSV VERIFIED")
print("============================================================")

                                   path  prediction
0  small_model_track_test/SM_test_0001/          36
1  small_model_track_test/SM_test_0002/           0
2  small_model_track_test/SM_test_0003/          10
3  small_model_track_test/SM_test_0004/           8
4  small_model_track_test/SM_test_0005/          10

Rows: 405
Columns: ['path', 'prediction']
Unique paths: 405
Prediction range: 0 to 39

✓ SUBMISSION CSV VERIFIED
